# Hallucination Detection on MedHallu — Experimental Pipeline

Priyamvada Nambiar, Ashwath Rao B
School of Computer Engineering, Manipal Institute of Technology

**Design.** Train on `pqa_artificial` (9,000 items → 18,000 answer
pairs). Test on the disjoint, expert-annotated `pqa_labeled` split
(1,000 items → 2,000 pairs). Question overlap verified as zero.

**How to read this notebook.** Every stage caches to Google Drive, so
cells report `cached` rather than retraining. Results in the paper
come from Cells 6–9.

**Superseded outputs.** Cells 4–5 (stage 6) and the bootstrap in
Cell 7 were replaced by the matched-protocol runs in Cell 6 and the
paired seed-level test reported in the paper. They are retained for
completeness and are not the reported results.

**Principal result — Cell 9.** A classifier given no evidence reaches
0.9520 macro F1 against 0.9650 with full evidence, and 0.9495 with
mismatched evidence.

In [ ]:
import torch, transformers, sklearn, datasets, numpy, pandas, platform
print("python      ", platform.python_version())
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("sklearn     ", sklearn.__version__)
print("datasets    ", datasets.__version__)
print("numpy       ", numpy.__version__, "| pandas", pandas.__version__)
print("GPU         ", torch.cuda.get_device_name(0))
print("\nSeeds: 42, 123, 456, 789, 1234")
print("Note: fp16 training on GPU is not bit-reproducible; repeated")
print("runs vary by roughly the initialisation SD reported in the paper.")

python       3.13.15
torch        2.11.0+cu128
transformers 5.16.1
sklearn      1.6.1
datasets     4.8.5
numpy        2.1.3 | pandas 2.2.3
GPU          Tesla T4

Seeds: 42, 123, 456, 789, 1234
Note: fp16 training on GPU is not bit-reproducible; repeated
runs vary by roughly the initialisation SD reported in the paper.


In [ ]:
import os
os.rename(f"{DRIVE}/results_v3.json", f"{DRIVE}/results_v3_SUPERSEDED.json")

In [1]:
# ============================================================
# CELL 1 — Setup. Run this first, every session.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, torch
DRIVE = "/content/drive/MyDrive/medhallu_v2"
os.makedirs(DRIVE, exist_ok=True)

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("Files on Drive:")
for f in sorted(os.listdir(DRIVE)):
    print("  ", f)

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Files on Drive:
   _touch.txt
   all_encoder_predictions.npz
   all_encoder_seeds.csv
   biomedbert_v2
   biomedbert_v2_tok
   external_halueval.json
   external_halueval_preds.csv
   figure_data_final.npz
   figure_data_v2.npz
   figure_data_v4.npz
   multiseed_v2.csv
   nli_test.csv
   nli_train.csv
   paired_bootstrap.json
   paired_seed_tests.json
   phase1_probs.npz
   results_v2.json
   results_v3_SUPERSEDED.json
   review_v14
   severity_final.csv
   severity_v2.csv
   severity_v3.csv
   severity_v4.csv
   shortcut_answer_only.json
   shortcut_answer_only_probs.npy
   shortcut_evidence_only.json
   shortcut_evidence_only_probs.npy
   shortcut_full.json
   shortcut_full_probs.npy
   shortcut_shuffled.json
   shortcut_shuffled_probs.npy
   shortcut_summary.json
   test_df.csv
   test_predictions.csv
   train_df.csv
   type_cv_cis.json
   type_cv_predictions.csv
   type_model_v2
   type_model_v3
   type_model_v4
   type_predictio

In [ ]:
# ============================================================
# CELL 2 - Data prep, Phase 1, NLI, ensemble, type classifier.
#          Definitions only. Nothing runs yet.
# ============================================================
import os, json, random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

DRIVE = "/content/drive/MyDrive/medhallu_v2"
os.makedirs(DRIVE, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BIOMEDBERT = ("microsoft/BiomedNLP-BiomedBERT-base-"
              "uncased-abstract-fulltext")
NLI_MODEL = "cross-encoder/nli-deberta-v3-large"

TYPES = ['Misinterpretation of #Question#',
         'Incomplete Information',
         'Mechanism and Pathway Misattribution',
         'Methodological and Evidence Fabrication']
TYPE2ID = {t: i for i, t in enumerate(TYPES)}


def stage_done(name):
    return os.path.exists(f"{DRIVE}/{name}")


# =====================================================================
# STAGE 0 — build dataframes, verify no question overlap
# =====================================================================
def build_data():
    if stage_done("train_df.csv") and stage_done("test_df.csv"):
        print("stage 0: loading cached")
        return (pd.read_csv(f"{DRIVE}/train_df.csv"),
                pd.read_csv(f"{DRIVE}/test_df.csv"))

    def to_rows(ds):
        rows = []
        for ex in ds:
            k = (" ".join(ex['Knowledge'])
                 if isinstance(ex['Knowledge'], list) else str(ex['Knowledge']))
            base = {'question': ex['Question'], 'knowledge': k,
                    'difficulty': ex['Difficulty Level'],
                    'htype': ex['Category of Hallucination']}
            rows.append({**base, 'answer': ex['Ground Truth'], 'label': 0})
            rows.append({**base, 'answer': ex['Hallucinated Answer'], 'label': 1})
        return pd.DataFrame(rows)

    art = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_artificial")['train']
    lab = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")['train']

    train_df = to_rows(art)
    test_df = to_rows(lab)

    # CRITICAL CHECK — the two splits must not share questions.
    overlap = set(train_df['question']) & set(test_df['question'])
    print(f"question overlap between splits: {len(overlap)}")
    if overlap:
        print("!! removing overlapping questions from TRAIN")
        train_df = train_df[~train_df['question'].isin(overlap)]
        train_df = train_df.reset_index(drop=True)

    train_df.to_csv(f"{DRIVE}/train_df.csv", index=False)
    test_df.to_csv(f"{DRIVE}/test_df.csv", index=False)
    print(f"train {len(train_df)} pairs | test {len(test_df)} pairs")
    return train_df, test_df


# =====================================================================
# STAGE 1 — BiomedBERT binary classifier
# =====================================================================
class PairDS(torch.utils.data.Dataset):
    def __init__(self, df, tok, label_col='label'):
        self.df = df.reset_index(drop=True); self.tok = tok
        self.label_col = label_col
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        enc = self.tok(f"{r['question']} {r['knowledge']}", r['answer'],
                       truncation='longest_first', max_length=512,
                       padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(int(r[self.label_col]))
        return item


def metrics_fn(p):
    preds = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, preds, average='macro')}


def train_phase1(train_df, test_df):
    if stage_done("phase1_probs.npz"):
        print("stage 1: loading cached")
        d = np.load(f"{DRIVE}/phase1_probs.npz")
        return d['train_p1'], d['test_p1']

    tok = AutoTokenizer.from_pretrained(BIOMEDBERT)
    model = AutoModelForSequenceClassification.from_pretrained(
        BIOMEDBERT, num_labels=2)

    args = TrainingArguments(
        output_dir="/content/p1_out",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="no",
        warmup_steps=500, logging_steps=200,
        fp16=True, seed=SEED, report_to="none")

    tr = Trainer(model=model, args=args,
                 train_dataset=PairDS(train_df, tok),
                 eval_dataset=PairDS(test_df, tok),
                 compute_metrics=metrics_fn)
    tr.train()

    model.save_pretrained(f"{DRIVE}/biomedbert_v2")
    tok.save_pretrained(f"{DRIVE}/biomedbert_v2_tok")

    train_p1 = torch.softmax(torch.tensor(
        tr.predict(PairDS(train_df, tok)).predictions), -1).numpy()[:, 1]
    test_p1 = torch.softmax(torch.tensor(
        tr.predict(PairDS(test_df, tok)).predictions), -1).numpy()[:, 1]

    np.savez(f"{DRIVE}/phase1_probs.npz",
             train_p1=train_p1, test_p1=test_p1)
    return train_p1, test_p1


# =====================================================================
# STAGE 2 — DeBERTa NLI scores  (slowest stage)
# =====================================================================
def nli_scores(df, tag, batch=64):
    path = f"{DRIVE}/nli_{tag}.csv"
    if os.path.exists(path):
        print(f"stage 2 [{tag}]: loading cached")
        return pd.read_csv(path)

    tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        NLI_MODEL).cuda().eval().half()

    # label order for this checkpoint: contradiction, entailment, neutral
    id2label = model.config.id2label
    print("NLI label map:", id2label)

    out = []
    with torch.no_grad():
        for i in range(0, len(df), batch):
            chunk = df.iloc[i:i+batch]
            enc = tok(list(chunk['knowledge']), list(chunk['answer']),
                      truncation=True, max_length=512,
                      padding=True, return_tensors='pt').to('cuda')
            probs = torch.softmax(model(**enc).logits.float(), -1).cpu().numpy()
            out.append(probs)
            if i % (batch * 50) == 0:
                print(f"  {tag}: {i}/{len(df)}")

    probs = np.vstack(out)
    cols = {id2label[j].lower(): probs[:, j] for j in range(probs.shape[1])}
    res = pd.DataFrame(cols)
    res['phi'] = res['contradiction'] - res['entailment']
    res.to_csv(path, index=False)
    return res


# =====================================================================
# STAGE 3 — gradient boosting ensemble
# =====================================================================
def train_ensemble(train_df, test_df, train_p1, test_p1, tr_nli, te_nli):
    feats = ['contradiction', 'neutral', 'entailment', 'phi']
    X_tr = np.column_stack([train_p1] + [tr_nli[c].values for c in feats])
    X_te = np.column_stack([test_p1] + [te_nli[c].values for c in feats])

    sc = StandardScaler().fit(X_tr)
    ens = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                     max_depth=3, random_state=SEED)
    ens.fit(sc.transform(X_tr), train_df['label'].values)

    pred = ens.predict(sc.transform(X_te))
    proba = ens.predict_proba(sc.transform(X_te))[:, 1]

    res = test_df[['label', 'difficulty', 'htype']].copy()
    res['phase1_pred'] = (test_p1 >= 0.5).astype(int)
    res['phase1_proba'] = test_p1
    res['phase2_pred'] = pred
    res['phase2_proba'] = proba
    res.to_csv(f"{DRIVE}/test_predictions.csv", index=False)
    return res


# =====================================================================
# STAGE 4 — type classifier (now with real support per class)
# =====================================================================
def train_type_classifier(train_df, test_df):
    if stage_done("type_predictions.csv"):
        print("stage 4: loading cached")
        return pd.read_csv(f"{DRIVE}/type_predictions.csv")

    tr = train_df[train_df['label'] == 1].copy()
    te = test_df[test_df['label'] == 1].copy()
    tr['tid'] = tr['htype'].map(TYPE2ID)
    te['tid'] = te['htype'].map(TYPE2ID)
    tr = tr.dropna(subset=['tid']); te = te.dropna(subset=['tid'])
    print("type train support:", tr['htype'].value_counts().to_dict())

    tok = AutoTokenizer.from_pretrained(BIOMEDBERT)
    model = AutoModelForSequenceClassification.from_pretrained(
        BIOMEDBERT, num_labels=4)

    counts = tr['tid'].value_counts().sort_index().values
    w = torch.tensor(len(tr) / (4 * counts), dtype=torch.float).cuda()

    class WTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            labels = inputs.pop("labels")
            out = model(**inputs)
            loss = torch.nn.functional.cross_entropy(out.logits, labels,
                                                     weight=w)
            return (loss, out) if return_outputs else loss

    args = TrainingArguments(
        output_dir="/content/type_out", num_train_epochs=4,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="no",
        warmup_steps=300, fp16=True, seed=SEED, report_to="none")

    t = WTrainer(model=model, args=args,
                 train_dataset=PairDS(tr, tok, 'tid'),
                 eval_dataset=PairDS(te, tok, 'tid'))
    t.train()

    preds = t.predict(PairDS(te, tok, 'tid')).predictions.argmax(-1)
    out = te[['htype', 'difficulty']].copy()
    out['true_tid'] = te['tid'].astype(int).values
    out['pred_tid'] = preds
    out.to_csv(f"{DRIVE}/type_predictions.csv", index=False)
    model.save_pretrained(f"{DRIVE}/type_model_v2")
    return out


# =====================================================================
# RUN
# =====================================================================

In [ ]:
# ============================================================
# CELL 3 — Run the main pipeline.  ~1 hour first time, instant if cached.
# ============================================================
train_df, test_df = build_data()
train_p1, test_p1 = train_phase1(train_df, test_df)
tr_nli = nli_scores(train_df, "train")
te_nli = nli_scores(test_df, "test")
res = train_ensemble(train_df, test_df, train_p1, test_p1, tr_nli, te_nli)

from sklearn.metrics import f1_score, roc_auc_score
y = res['label'].values
hard = (res['difficulty'] == 'hard').values
for name, col in [("Phase 1", 'phase1_pred'), ("Phase 2", 'phase2_pred')]:
    print(f"{name}: macro F1 {f1_score(y, res[col], average='macro'):.4f} "
          f"| hard {f1_score(y[hard], res[col][hard], average='macro'):.4f}")
print(f"AUROC {roc_auc_score(y, res['phase2_proba']):.4f}")

stage 0: loading cached
stage 1: loading cached
stage 2 [train]: loading cached
stage 2 [test]: loading cached
Phase 1: macro F1 0.9715 | hard 0.9706
Phase 2: macro F1 0.9715 | hard 0.9718
AUROC 0.9944


In [ ]:
# ============================================================
# CELL 4 - Multi-seed and encoder baseline definitions.
# ============================================================
import os, json
import numpy as np, pandas as pd, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.metrics import classification_report
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

DRIVE = "/content/drive/MyDrive/medhallu_v2"
BIOMEDBERT = ("microsoft/BiomedNLP-BiomedBERT-base-"
              "uncased-abstract-fulltext")
TYPES = ['Misinterpretation of #Question#', 'Incomplete Information',
         'Mechanism and Pathway Misattribution',
         'Methodological and Evidence Fabrication']
TYPE2ID = {t: i for i, t in enumerate(TYPES)}
SHORT = ['Misinterpretation', 'Incomplete info',
         'Mechanism misattr.', 'Evidence fabric.']

train_df = pd.read_csv(f"{DRIVE}/train_df.csv")
test_df  = pd.read_csv(f"{DRIVE}/test_df.csv")
tr_nli   = pd.read_csv(f"{DRIVE}/nli_train.csv")
te_nli   = pd.read_csv(f"{DRIVE}/nli_test.csv")

y_tr = train_df['label'].values
y_te = test_df['label'].values
NLI_F = ['contradiction', 'neutral', 'entailment', 'phi']

results = json.load(open(f"{DRIVE}/results_v2.json"))


class PairDS(torch.utils.data.Dataset):
    def __init__(self, df, tok, col='label'):
        self.df = df.reset_index(drop=True); self.tok = tok; self.col = col
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        enc = self.tok(f"{r['question']} {r['knowledge']}", r['answer'],
                       truncation='longest_first', max_length=512,
                       padding='max_length', return_tensors='pt')
        it = {k: v.squeeze(0) for k, v in enc.items()}
        it['labels'] = torch.tensor(int(r[self.col]))
        return it


def macro_metrics(p):
    return {'macro_f1': f1_score(p.label_ids, p.predictions.argmax(-1),
                                 average='macro')}


# =====================================================================
# JOB 0 — type classifier, model selection on macro F1
# =====================================================================
def job0():
    hall_tr = train_df[train_df['label'] == 1].copy()
    hall_tr['tid'] = hall_tr['htype'].map(TYPE2ID)
    hall_tr = hall_tr.dropna(subset=['tid'])

    qs = hall_tr['question'].unique()
    np.random.default_rng(42).shuffle(qs)
    cut = int(0.85 * len(qs))
    art_train = hall_tr[hall_tr['question'].isin(set(qs[:cut]))]
    art_held  = hall_tr[hall_tr['question'].isin(set(qs[cut:]))]

    hall_te = test_df[test_df['label'] == 1].copy()
    hall_te['tid'] = hall_te['htype'].map(TYPE2ID)
    hall_te = hall_te.dropna(subset=['tid'])

    tok = AutoTokenizer.from_pretrained(BIOMEDBERT)
    model = AutoModelForSequenceClassification.from_pretrained(
        BIOMEDBERT, num_labels=4)

    counts = art_train['tid'].value_counts().sort_index().values
    # sqrt-tempered weights: full inverse-frequency was over-correcting
    # and destroying recall on the dominant class.
    w_raw = len(art_train) / (4 * counts)
    w = torch.tensor(np.sqrt(w_raw), dtype=torch.float).cuda()
    print("class weights:", w.cpu().numpy().round(3))

    class WTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            labels = inputs.pop("labels")
            o = model(**inputs)
            loss = torch.nn.functional.cross_entropy(o.logits, labels,
                                                     weight=w)
            return (loss, o) if return_outputs else loss

    args = TrainingArguments(
        output_dir="/content/type_out3", num_train_epochs=5,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1", greater_is_better=True,
        save_total_limit=1, warmup_steps=300,
        fp16=True, seed=42, report_to="none")

    t = WTrainer(model=model, args=args,
                 train_dataset=PairDS(art_train, tok, 'tid'),
                 eval_dataset=PairDS(art_held, tok, 'tid'),
                 compute_metrics=macro_metrics)
    t.train()

    out = {}
    for df, name in [(hall_te, 'labeled'), (art_held, 'artificial')]:
        pred = t.predict(PairDS(df, tok, 'tid')).predictions.argmax(-1)
        true = df['tid'].astype(int).values
        print(f"\n=== JOB 0 type: {name} ===")
        print(classification_report(true, pred, target_names=SHORT,
                                    zero_division=0, digits=4))
        rep = classification_report(true, pred, target_names=SHORT,
                                    zero_division=0, output_dict=True)
        d = df[['htype', 'difficulty']].copy()
        d['true_tid'] = true; d['pred_tid'] = pred
        d.to_csv(f"{DRIVE}/type_predictions_{name}_v3.csv", index=False)
        out[name] = {'macro_f1': round(rep['macro avg']['f1-score'], 4),
                     'per_class': {s: {k: round(rep[s][k], 4)
                                       for k in ['precision', 'recall',
                                                 'f1-score']}
                                   | {'n': int(rep[s]['support'])}
                                   for s in SHORT}}
    model.save_pretrained(f"{DRIVE}/type_model_v4")
    return out


# =====================================================================
# JOB A — multi-seed. Split is fixed; this isolates initialisation.
# =====================================================================
def run_encoder(model_name, seed, tag):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2)
    args = TrainingArguments(
        output_dir=f"/content/{tag}_out", num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="no", save_strategy="no",
        warmup_steps=500, logging_steps=500,
        fp16=True, seed=seed, report_to="none")
    t = Trainer(model=model, args=args,
                train_dataset=PairDS(train_df, tok))
    t.train()
    tr_p = torch.softmax(torch.tensor(
        t.predict(PairDS(train_df, tok)).predictions), -1).numpy()[:, 1]
    te_p = torch.softmax(torch.tensor(
        t.predict(PairDS(test_df, tok)).predictions), -1).numpy()[:, 1]
    del model, t; torch.cuda.empty_cache()
    return tr_p, te_p


def fit_ensemble(tr_p, te_p, seed):
    X_tr = np.column_stack([tr_p] + [tr_nli[c].values for c in NLI_F])
    X_te = np.column_stack([te_p] + [te_nli[c].values for c in NLI_F])
    sc = StandardScaler().fit(X_tr)
    g = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                   max_depth=3, random_state=seed)
    g.fit(sc.transform(X_tr), y_tr)
    return (g.predict(sc.transform(X_te)),
            g.predict_proba(sc.transform(X_te))[:, 1])


def jobA():
    rows = []
    hard = (test_df['difficulty'] == 'hard').values
    for seed in [42, 123, 456, 789, 1234]:
        print(f"\n--- JOB A seed {seed} ---")
        tr_p, te_p = run_encoder(BIOMEDBERT, seed, f"ms{seed}")
        pred, proba = fit_ensemble(tr_p, te_p, seed)
        rows.append({
            'seed': seed,
            'macro_f1': round(f1_score(y_te, pred, average='macro'), 4),
            'hard_f1': round(f1_score(y_te[hard], pred[hard],
                                      average='macro'), 4),
            'auroc': round(roc_auc_score(y_te, proba), 4)})
        print(rows[-1])
    df = pd.DataFrame(rows)
    df.to_csv(f"{DRIVE}/multiseed_v2.csv", index=False)
    summ = {}
    for c in ['macro_f1', 'hard_f1', 'auroc']:
        v = df[c].values
        m, s = v.mean(), v.std(ddof=1)
        ci = 1.96 * s / np.sqrt(len(v))
        summ[c] = {'mean': round(m, 4), 'std': round(s, 4),
                   'ci95': [round(m-ci, 4), round(m+ci, 4)]}
    print(json.dumps(summ, indent=2))
    return {'per_seed': rows, 'summary': summ}


# =====================================================================
# JOB B — encoder baselines, same split, Phase 1 style (no ensemble)
# =====================================================================
def jobB():
    models = {
        'RoBERTa': 'roberta-base',
        'BioBERT': 'dmis-lab/biobert-base-cased-v1.2',
        'Bio_ClinicalBERT': 'emilyalsentzer/Bio_ClinicalBERT',
    }
    hard = (test_df['difficulty'] == 'hard').values
    out = {}
    for name, path in models.items():
        print(f"\n--- JOB B {name} ---")
        try:
            _, te_p = run_encoder(path, 42, name.lower())
            pred = (te_p >= 0.5).astype(int)
            out[name] = {
                'macro_f1': round(f1_score(y_te, pred, average='macro'), 4),
                'hard_f1': round(f1_score(y_te[hard], pred[hard],
                                          average='macro'), 4),
                'auroc': round(roc_auc_score(y_te, te_p), 4),
                'auprc': round(average_precision_score(y_te, te_p), 4)}
            print(name, out[name])
        except Exception as e:
            print(f"{name} FAILED: {e}")
            out[name] = {'error': str(e)}
    return out

In [ ]:
# ============================================================
# CELL 5 — Multi-seed and baselines (superseded)
#
# The stage-6 versions of these experiments were replaced by the
# stage-7 runs in Cell 7, which train all four encoders under one
# matched protocol across five seeds. The results reported in the
# paper come from Cell 7. Stored stage-6 output is loaded below
# for completeness.
# ============================================================
import json, pandas as pd
print(pd.read_csv(f"{DRIVE}/multiseed_v2.csv"))
print(json.dumps(json.load(open(f"{DRIVE}/results_v3.json"))['baselines_v2'],
                 indent=2))

   seed  macro_f1  hard_f1   auroc
0    42    0.9700   0.9694  0.9932
1   123    0.9675   0.9694  0.9928
2   456    0.9665   0.9669  0.9923
3   789    0.9725   0.9706  0.9900
4  1234    0.9715   0.9706  0.9924
{
  "RoBERTa": {
    "macro_f1": 0.959,
    "hard_f1": 0.9546,
    "auroc": 0.9939,
    "auprc": 0.9944
  },
  "BioBERT": {
    "macro_f1": 0.9635,
    "hard_f1": 0.9706,
    "auroc": 0.9934,
    "auprc": 0.9936
  },
  "Bio_ClinicalBERT": {
    "macro_f1": 0.9625,
    "hard_f1": 0.9632,
    "auroc": 0.9936,
    "auprc": 0.9941
  }
}


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/medhallu_v2"
for f in ["multiseed_v2.csv", "all_encoder_seeds.csv",
          "all_encoder_predictions.npz", "type_cv_predictions.csv",
          "shortcut_summary.json", "external_halueval.json",
          "results_v3.json"]:
    p = f"{DRIVE}/{f}"
    print(f"{f:32s}", "OK" if os.path.exists(p) else "MISSING")

multiseed_v2.csv                 OK
all_encoder_seeds.csv            OK
all_encoder_predictions.npz      OK
type_cv_predictions.csv          OK
shortcut_summary.json            OK
external_halueval.json           OK
results_v3.json                  OK


In [ ]:
# ============================================================
# CELL 6 - All-encoder seeds, paired test, type CV, external transfer.
# ============================================================
import os, json, itertools
import numpy as np, pandas as pd, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import (f1_score, roc_auc_score, classification_report,
                             average_precision_score, confusion_matrix)
from sklearn.model_selection import GroupKFold

DRIVE = "/content/drive/MyDrive/medhallu_v2"
BIOMEDBERT = ("microsoft/BiomedNLP-BiomedBERT-base-"
              "uncased-abstract-fulltext")
ENCODERS = {
    'BiomedBERT':       BIOMEDBERT,
    'BioBERT':          'dmis-lab/biobert-base-cased-v1.2',
    'Bio_ClinicalBERT': 'emilyalsentzer/Bio_ClinicalBERT',
    'RoBERTa':          'roberta-base',
}
SEEDS = [42, 123, 456, 789, 1234]
TYPES = ['Misinterpretation of #Question#', 'Incomplete Information',
         'Mechanism and Pathway Misattribution',
         'Methodological and Evidence Fabrication']
TYPE2ID = {t: i for i, t in enumerate(TYPES)}
SHORT = ['Misinterpretation', 'Incomplete info',
         'Mechanism misattr.', 'Evidence fabric.']

train_df = pd.read_csv(f"{DRIVE}/train_df.csv")
test_df  = pd.read_csv(f"{DRIVE}/test_df.csv")
y_te = test_df['label'].values
hard = (test_df['difficulty'] == 'hard').values


class PairDS(torch.utils.data.Dataset):
    def __init__(self, df, tok, col='label'):
        self.df = df.reset_index(drop=True); self.tok = tok; self.col = col
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        enc = self.tok(f"{r['question']} {r['knowledge']}", r['answer'],
                       truncation='longest_first', max_length=512,
                       padding='max_length', return_tensors='pt')
        it = {k: v.squeeze(0) for k, v in enc.items()}
        it['labels'] = torch.tensor(int(r[self.col]))
        return it


def train_and_predict(model_name, seed, train_data, predict_data,
                      n_labels=2, epochs=3, label_col='label',
                      class_weights=None):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=n_labels)

    if class_weights is not None:
        w = torch.tensor(class_weights, dtype=torch.float).cuda()
        class T(Trainer):
            def compute_loss(self, model, inputs, return_outputs=False, **kw):
                lab = inputs.pop("labels")
                o = model(**inputs)
                loss = torch.nn.functional.cross_entropy(o.logits, lab, weight=w)
                return (loss, o) if return_outputs else loss
        TrainerCls = T
    else:
        TrainerCls = Trainer

    args = TrainingArguments(
        output_dir=f"/content/tmp_{seed}", num_train_epochs=epochs,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="no", save_strategy="no",
        warmup_steps=500, logging_steps=1000,
        fp16=True, seed=seed, report_to="none")
    t = TrainerCls(model=model, args=args,
                   train_dataset=PairDS(train_data, tok, label_col))
    t.train()
    logits = t.predict(PairDS(predict_data, tok, label_col)).predictions
    probs = torch.softmax(torch.tensor(logits), -1).numpy()
    del model, t; torch.cuda.empty_cache()
    return probs


# =====================================================================
# JOB A — all encoders x 5 seeds
# =====================================================================
def jobA():
    path = f"{DRIVE}/all_encoder_seeds.csv"
    pred_path = f"{DRIVE}/all_encoder_predictions.npz"
    if os.path.exists(path) and os.path.exists(pred_path):
        print("JOB A cached"); return pd.read_csv(path)

    rows, preds = [], {}
    for name, mpath in ENCODERS.items():
        for seed in SEEDS:
            print(f"\n--- JOB A {name} seed {seed} ---", flush=True)
            p = train_and_predict(mpath, seed, train_df, test_df)[:, 1]
            pred = (p >= 0.5).astype(int)
            preds[f"{name}_{seed}"] = p
            rows.append({'encoder': name, 'seed': seed,
                         'macro_f1': f1_score(y_te, pred, average='macro'),
                         'hard_f1': f1_score(y_te[hard], pred[hard],
                                             average='macro'),
                         'auroc': roc_auc_score(y_te, p),
                         'auprc': average_precision_score(y_te, p)})
            print(rows[-1], flush=True)
            pd.DataFrame(rows).to_csv(path, index=False)
            np.savez(pred_path, **preds)

    df = pd.DataFrame(rows)
    print("\n=== JOB A summary ===")
    for enc in ENCODERS:
        d = df[df['encoder'] == enc]
        for m in ['macro_f1', 'hard_f1', 'auroc']:
            v = d[m].values; mu, sd = v.mean(), v.std(ddof=1)
            ci = 1.96 * sd / np.sqrt(len(v))
            print(f"{enc:18s} {m:9s} {mu:.4f} +/- {sd:.4f} "
                  f"CI [{mu-ci:.4f}, {mu+ci:.4f}]")
    return df


# =====================================================================
# JOB A2 — paired bootstrap between encoders
# =====================================================================
def jobA2(n_boot=10000, seed=0):
    preds = np.load(f"{DRIVE}/all_encoder_predictions.npz")
    rng = np.random.default_rng(seed)
    n = len(y_te)

    def mean_pred(enc):
        cols = [preds[f"{enc}_{s}"] for s in SEEDS if f"{enc}_{s}" in preds]
        return np.mean(cols, axis=0)

    out = {}
    base = 'BiomedBERT'
    pb = mean_pred(base)
    for other in [e for e in ENCODERS if e != base]:
        po = mean_pred(other)
        diffs = np.empty(n_boot)
        for b in range(n_boot):
            idx = rng.integers(0, n, n)
            diffs[b] = (f1_score(y_te[idx], (pb[idx] >= .5).astype(int),
                                 average='macro') -
                        f1_score(y_te[idx], (po[idx] >= .5).astype(int),
                                 average='macro'))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        p = float((diffs <= 0).mean() * 2)
        out[f"{base} vs {other}"] = {
            'mean_diff': round(float(diffs.mean()), 4),
            'ci95': [round(float(lo), 4), round(float(hi), 4)],
            'p_two_sided': round(min(p, 1.0), 4)}
        print(f"{base} vs {other}: {out[f'{base} vs {other}']}", flush=True)

    json.dump(out, open(f"{DRIVE}/paired_bootstrap.json", "w"), indent=2)
    return out


# =====================================================================
# JOB B — 5-fold CV type classification
#         Every Evidence Fabrication example gets predicted once.
# =====================================================================
def jobB():
    path = f"{DRIVE}/type_cv_predictions.csv"
    if os.path.exists(path):
        print("JOB B cached"); return pd.read_csv(path)

    hall = train_df[train_df['label'] == 1].copy()
    hall['tid'] = hall['htype'].map(TYPE2ID)
    hall = hall.dropna(subset=['tid']).reset_index(drop=True)
    print("total hallucinated:", len(hall))
    print(hall['htype'].value_counts().to_dict())

    gkf = GroupKFold(n_splits=5)
    all_true, all_pred, all_fold = [], [], []

    for fold, (tr, te) in enumerate(
            gkf.split(hall, groups=hall['question'])):
        print(f"\n--- JOB B fold {fold+1}/5 ---", flush=True)
        d_tr, d_te = hall.iloc[tr], hall.iloc[te]
        print("  fold test support:", d_te['htype'].value_counts().to_dict())
        counts = d_tr['tid'].value_counts().sort_index().values
        w = np.sqrt(len(d_tr) / (4 * counts))
        probs = train_and_predict(BIOMEDBERT, 42, d_tr, d_te,
                                  n_labels=4, epochs=2, label_col='tid',
                                  class_weights=w)
        all_true.extend(d_te['tid'].astype(int).values)
        all_pred.extend(probs.argmax(-1))
        all_fold.extend([fold] * len(d_te))

    res = pd.DataFrame({'true_tid': all_true, 'pred_tid': all_pred,
                        'fold': all_fold})
    res.to_csv(path, index=False)

    print("\n=== JOB B: pooled 5-fold type classification ===")
    print(classification_report(res['true_tid'], res['pred_tid'],
                                target_names=SHORT, zero_division=0,
                                digits=4))
    print("confusion matrix (rows true, cols predicted):")
    print(confusion_matrix(res['true_tid'], res['pred_tid']))

    # bootstrap CIs per class
    rng = np.random.default_rng(0)
    t, p = res['true_tid'].values, res['pred_tid'].values
    cis = {}
    for k, s in enumerate(SHORT):
        boots = []
        for _ in range(2000):
            idx = rng.integers(0, len(t), len(t))
            boots.append(f1_score(t[idx] == k, p[idx] == k, zero_division=0))
        cis[s] = [round(float(np.percentile(boots, 2.5)), 4),
                  round(float(np.percentile(boots, 97.5)), 4)]
    print("\nper-class F1 95% CI:", json.dumps(cis, indent=2))
    json.dump(cis, open(f"{DRIVE}/type_cv_cis.json", "w"), indent=2)
    return res


# =====================================================================
# JOB C — external evaluation, different generator and domain
# =====================================================================
def jobC():
    path = f"{DRIVE}/external_halueval.json"
    if os.path.exists(path):
        print("JOB C cached"); return json.load(open(path))

    from datasets import load_dataset
    ds = load_dataset("pminervini/HaluEval", "qa")['data']
    print("HaluEval QA:", len(ds), ds.column_names)

    rows = []
    for ex in ds.select(range(2000)):
        base = {'question': ex['question'], 'knowledge': ex['knowledge']}
        rows.append({**base, 'answer': ex['right_answer'], 'label': 0})
        rows.append({**base, 'answer': ex['hallucinated_answer'],
                     'label': 1})
    ext = pd.DataFrame(rows)
    print("external pairs:", len(ext))

    tok = AutoTokenizer.from_pretrained(f"{DRIVE}/biomedbert_v2_tok")
    model = AutoModelForSequenceClassification.from_pretrained(
        f"{DRIVE}/biomedbert_v2").cuda().eval()

    probs = []
    with torch.no_grad():
        for i in range(0, len(ext), 64):
            c = ext.iloc[i:i+64]
            enc = tok([f"{q} {k}" for q, k in zip(c['question'],
                                                  c['knowledge'])],
                      list(c['answer']), truncation='longest_first',
                      max_length=512, padding=True,
                      return_tensors='pt').to('cuda')
            probs.append(torch.softmax(model(**enc).logits, -1)
                         .cpu().numpy()[:, 1])
            if i % 640 == 0: print(f"  {i}/{len(ext)}", flush=True)

    p = np.concatenate(probs)
    yv = ext['label'].values
    pred = (p >= 0.5).astype(int)
    out = {'n_pairs': len(ext),
           'macro_f1': round(f1_score(yv, pred, average='macro'), 4),
           'auroc': round(float(roc_auc_score(yv, p)), 4),
           'auprc': round(float(average_precision_score(yv, p)), 4)}
    print("\n=== JOB C: HaluEval QA (zero-shot transfer) ===")
    print(json.dumps(out, indent=2))
    print("NOTE: HaluEval QA is built from HotpotQA, so this shifts both")
    print("generator AND domain. Interpret as a joint shift, not a clean")
    print("generator-only test.")
    ext['proba'] = p
    ext[['label', 'proba']].to_csv(f"{DRIVE}/external_halueval_preds.csv",
                                   index=False)
    json.dump(out, open(path, "w"), indent=2)
    return out

In [ ]:
# ============================================================
# CELL 7 — Run them.  ~2.5 hours first time.
# ============================================================
jobA()    # 4 encoders x 5 seeds
jobA2()   # paired comparison
jobB()    # 5-fold type cross-validation
jobC()    # HaluEval external transfer

JOB A cached
BiomedBERT vs BioBERT: {'mean_diff': 0.008, 'ci95': [0.0005, 0.0155], 'p_two_sided': 0.039}
BiomedBERT vs Bio_ClinicalBERT: {'mean_diff': 0.015, 'ci95': [0.007, 0.023], 'p_two_sided': 0.0}
BiomedBERT vs RoBERTa: {'mean_diff': 0.0125, 'ci95': [0.0045, 0.0205], 'p_two_sided': 0.0024}
JOB B cached
JOB C cached


{'n_pairs': 4000, 'macro_f1': 0.6109, 'auroc': 0.6914, 'auprc': 0.7387}

In [ ]:
# ============================================================
# CELL 8 - Shortcut diagnostics. THE PAPER'S MAIN RESULT.
#
#   full          question + knowledge || answer
#   answer_only   (empty)              || answer
#   evidence_only question + knowledge || (empty)     <- control
#   shuffled      ANOTHER item's evidence || answer
# ============================================================
import os, json
import numpy as np, pandas as pd, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import f1_score, roc_auc_score

DRIVE = "/content/drive/MyDrive/medhallu_v2"
BIOMEDBERT = ("microsoft/BiomedNLP-BiomedBERT-base-"
              "uncased-abstract-fulltext")
SEED = 42

train_df = pd.read_csv(f"{DRIVE}/train_df.csv")
test_df  = pd.read_csv(f"{DRIVE}/test_df.csv")
y_te = test_df['label'].values
hard = (test_df['difficulty'] == 'hard').values

# shuffled evidence: pair each row with another item's knowledge
rng = np.random.default_rng(SEED)
def shuffle_evidence(df):
    d = df.copy().reset_index(drop=True)
    perm = rng.permutation(len(d))
    # avoid accidentally keeping the true pairing
    same = perm == np.arange(len(d))
    perm[same] = (perm[same] + 1) % len(d)
    d['knowledge'] = d['knowledge'].values[perm]
    d['question']  = d['question'].values[perm]
    return d

CONDITIONS = {
    'full':          lambda d: d,
    'answer_only':   lambda d: d.assign(question='', knowledge=''),
    'evidence_only': lambda d: d.assign(answer=''),
    'shuffled':      shuffle_evidence,
}


class PairDS(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.df = df.reset_index(drop=True); self.tok = tok
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        a = f"{r['question']} {r['knowledge']}".strip() or "[EMPTY]"
        b = str(r['answer']).strip() or "[EMPTY]"
        enc = self.tok(a, b, truncation='longest_first', max_length=512,
                       padding='max_length', return_tensors='pt')
        it = {k: v.squeeze(0) for k, v in enc.items()}
        it['labels'] = torch.tensor(int(r['label']))
        return it


def run(name, fn):
    path = f"{DRIVE}/shortcut_{name}.json"
    if os.path.exists(path):
        print(f"{name}: cached"); return json.load(open(path))

    tr, te = fn(train_df), fn(test_df)
    tok = AutoTokenizer.from_pretrained(BIOMEDBERT)
    model = AutoModelForSequenceClassification.from_pretrained(
        BIOMEDBERT, num_labels=2)
    args = TrainingArguments(
        output_dir=f"/content/sc_{name}", num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="no", save_strategy="no", warmup_steps=500,
        logging_steps=1000, fp16=True, seed=SEED, report_to="none")
    t = Trainer(model=model, args=args, train_dataset=PairDS(tr, tok))
    t.train()
    p = torch.softmax(torch.tensor(
        t.predict(PairDS(te, tok)).predictions), -1).numpy()[:, 1]
    pred = (p >= .5).astype(int)
    res = {'condition': name,
           'macro_f1': round(f1_score(y_te, pred, average='macro'), 4),
           'hard_f1': round(f1_score(y_te[hard], pred[hard],
                                     average='macro'), 4),
           'auroc': round(float(roc_auc_score(y_te, p)), 4)}
    np.save(f"{DRIVE}/shortcut_{name}_probs.npy", p)
    json.dump(res, open(path, "w"), indent=2)
    del model, t; torch.cuda.empty_cache()
    print(res, flush=True)
    return res

In [ ]:
# ============================================================
# CELL 9 — Run shortcut diagnostics.  ~90 min on T4.
# ============================================================
import json
out = {}
for name, fn in CONDITIONS.items():
    print(f"\n===== {name} =====", flush=True)
    out[name] = run(name, fn)

json.dump(out, open(f"{DRIVE}/shortcut_summary.json", "w"), indent=2)

print("\n" + "=" * 58)
print(f"{'condition':16s} {'macro F1':>9s} {'hard F1':>9s} {'AUROC':>9s}")
for n, r in out.items():
    print(f"{n:16s} {r['macro_f1']:>9.4f} {r['hard_f1']:>9.4f} {r['auroc']:>9.4f}")
print("=" * 58)
full = out['full']['macro_f1']
print(f"answer_only retains {out['answer_only']['macro_f1']/full*100:.1f}% of full")
print(f"shuffled    retains {out['shuffled']['macro_f1']/full*100:.1f}% of full")
print(f"evidence_only control (must be ~0.50 AUROC): {out['evidence_only']['auroc']:.4f}")


===== full =====
full: cached

===== answer_only =====
answer_only: cached

===== evidence_only =====
evidence_only: cached

===== shuffled =====
shuffled: cached

condition         macro F1   hard F1     AUROC
full                0.9650    0.9620    0.9939
answer_only         0.9520    0.9485    0.9927
evidence_only       0.3333    0.3333    0.5000
shuffled            0.9495    0.9399    0.9925
answer_only retains 98.7% of full
shuffled    retains 98.4% of full
evidence_only control (must be ~0.50 AUROC): 0.5000


In [ ]:
# ============================================================
# CELL 10 — Print every stored result.
# ============================================================
import json, os
for f in ["results_v3.json", "shortcut_summary.json",
          "external_halueval.json", "paired_bootstrap.json",
          "type_cv_cis.json"]:
    p = f"{DRIVE}/{f}"
    if os.path.exists(p):
        print(f"\n########## {f} ##########")
        print(json.dumps(json.load(open(p)), indent=2))
    else:
        print(f"\n########## {f} — NOT PRESENT ##########")



########## results_v3.json ##########
{
  "phase1": {
    "macro_f1": 0.9715,
    "sensitivity": 0.969,
    "specificity": 0.974,
    "precision": 0.9739,
    "auroc": 0.9961,
    "auprc": 0.9965
  },
  "phase2": {
    "macro_f1": 0.974,
    "sensitivity": 0.963,
    "specificity": 0.985,
    "precision": 0.9847,
    "auroc": 0.9936,
    "auprc": 0.9937
  },
  "per_difficulty": {
    "easy": {
      "n": 548,
      "phase1": 0.9671,
      "phase2": 0.9708
    },
    "medium": {
      "n": 636,
      "phase1": 0.9717,
      "phase2": 0.9764
    },
    "hard": {
      "n": 816,
      "phase1": 0.9743,
      "phase2": 0.9743
    }
  },
  "random_baseline": 0.496,
  "nli_only": 0.6919,
  "explainability": {
    "mean_hall": 0.3057,
    "mean_clean": 0.0726,
    "cohens_d": 0.6666,
    "mannwhitney_p": "4.83e-54"
  },
  "_note": "phase1/phase2/per_difficulty carried over from the first run",
  "type_v3": {
    "labeled": {
      "macro_f1": 0.4531,
      "per_class": {
        "Misinterpre

In [ ]:
print("TABLE II — encoders, 5 seeds (macro F1, mean ± SD)")
print("  BiomedBERT        0.9674 ± 0.0045")
print("  BioBERT           0.9613 ± 0.0040")
print("  RoBERTa           0.9587 ± 0.0030")
print("  Bio_ClinicalBERT  0.9554 ± 0.0039")
print("\nTABLE III — paired t test across matched seeds, Bonferroni")
print("  vs BioBERT           +0.0061   p_adj 0.469   not significant")
print("  vs RoBERTa           +0.0087   p_adj 0.031")
print("  vs Bio_ClinicalBERT  +0.0120   p_adj 0.050")
print("\nSHORTCUT DIAGNOSTICS — principal result")
print("  full             0.9650")
print("  answer_only      0.9520   (98.7% of full, no evidence)")
print("  shuffled         0.9495   (98.4% of full, wrong evidence)")
print("  evidence_only    0.3333   (control, AUROC 0.500)")
print("\nTRANSFER — HaluEval QA: 0.6109 macro F1, 0.6914 AUROC")

TABLE II — encoders, 5 seeds (macro F1, mean ± SD)
  BiomedBERT        0.9674 ± 0.0045
  BioBERT           0.9613 ± 0.0040
  RoBERTa           0.9587 ± 0.0030
  Bio_ClinicalBERT  0.9554 ± 0.0039

TABLE III — paired t test across matched seeds, Bonferroni
  vs BioBERT           +0.0061   p_adj 0.469   not significant
  vs RoBERTa           +0.0087   p_adj 0.031
  vs Bio_ClinicalBERT  +0.0120   p_adj 0.050

SHORTCUT DIAGNOSTICS — principal result
  full             0.9650
  answer_only      0.9520   (98.7% of full, no evidence)
  shuffled         0.9495   (98.4% of full, wrong evidence)
  evidence_only    0.3333   (control, AUROC 0.500)

TRANSFER — HaluEval QA: 0.6109 macro F1, 0.6914 AUROC


In [ ]:
# ============================================================
# CELL 11 — Paired seed-level comparison (Table III in the paper)
#
# Supersedes the bootstrap in Cell 7. The bootstrap resamples the
# 2000 test pairs as if independent, but each MedHallu item yields
# one clean and one hallucinated answer sharing a question and
# passage; it also ignores training-seed variability and applies no
# multiplicity correction. The four encoders were trained under
# matched seeds, so a paired test across those seeds is appropriate.
# ============================================================
import numpy as np, pandas as pd, json
from scipy import stats

df = pd.read_csv(f"{DRIVE}/all_encoder_seeds.csv")
SEEDS = [42, 123, 456, 789, 1234]

piv = df.pivot(index='seed', columns='encoder', values='macro_f1').loc[SEEDS]
print("per-seed macro F1\n", piv.round(4), "\n")

print("TABLE II  mean +/- SD, t4 95% CI")
summary = {}
for enc in piv.columns:
    v = piv[enc].values
    m, s = v.mean(), v.std(ddof=1)
    ci = stats.t.ppf(.975, len(v)-1) * s / np.sqrt(len(v))
    summary[enc] = {'mean': round(float(m), 4), 'sd': round(float(s), 4),
                    'ci95': [round(float(m-ci), 4), round(float(m+ci), 4)]}
    print(f"  {enc:18s} {m:.4f} +/- {s:.4f}   [{m-ci:.4f}, {m+ci:.4f}]")

print("\nTABLE III  paired t test vs BiomedBERT, Bonferroni over 3 tests")
base = piv['BiomedBERT'].values
paired = {}
for enc in [c for c in piv.columns if c != 'BiomedBERT']:
    d = base - piv[enc].values
    t, p = stats.ttest_rel(base, piv[enc].values)
    ci = stats.t.ppf(.975, len(d)-1) * d.std(ddof=1) / np.sqrt(len(d))
    padj = min(p * 3, 1.0)
    paired[enc] = {'mean_diff': round(float(d.mean()), 4),
                   'ci95': [round(float(d.mean()-ci), 4),
                            round(float(d.mean()+ci), 4)],
                   'p': round(float(p), 4), 'p_adj': round(float(padj), 4)}
    flag = "" if padj < 0.05 else "   NOT SIGNIFICANT"
    print(f"  vs {enc:18s} {d.mean():+.4f} "
          f"[{d.mean()-ci:+.4f}, {d.mean()+ci:+.4f}] "
          f"p={p:.3f} p_adj={padj:.3f}{flag}")

json.dump({'table2': summary, 'table3': paired},
          open(f"{DRIVE}/paired_seed_tests.json", "w"), indent=2)

per-seed macro F1
 encoder  BioBERT  Bio_ClinicalBERT  BiomedBERT  RoBERTa
seed                                                   
42        0.9650            0.9510      0.9630   0.9560
123       0.9595            0.9565      0.9700   0.9560
456       0.9655            0.9585      0.9675   0.9625
789       0.9560            0.9515      0.9735   0.9610
1234      0.9605            0.9595      0.9630   0.9580 

TABLE II  mean +/- SD, t4 95% CI
  BioBERT            0.9613 +/- 0.0040   [0.9564, 0.9662]
  Bio_ClinicalBERT   0.9554 +/- 0.0039   [0.9505, 0.9603]
  BiomedBERT         0.9674 +/- 0.0045   [0.9618, 0.9730]
  RoBERTa            0.9587 +/- 0.0030   [0.9550, 0.9624]

TABLE III  paired t test vs BiomedBERT, Bonferroni over 3 tests
  vs BioBERT            +0.0061 [-0.0036, +0.0158] p=0.156 p_adj=0.469   NOT SIGNIFICANT
  vs Bio_ClinicalBERT   +0.0120 [+0.0036, +0.0204] p=0.017 p_adj=0.050
  vs RoBERTa            +0.0087 [+0.0034, +0.0140] p=0.010 p_adj=0.031


In [ ]:
# ============================================================
# CELL 12 — Severity and contradiction analysis (paper Section VII)
#
# Uses the seed 42 predictions from Cell 6's matched-protocol run,
# so these figures share one execution with Table II.
# ============================================================
import numpy as np, pandas as pd, json
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.metrics import (f1_score, roc_auc_score, recall_score,
                             precision_score, average_precision_score)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict

test_df = pd.read_csv(f"{DRIVE}/test_df.csv")
te_nli  = pd.read_csv(f"{DRIVE}/nli_test.csv")
tp      = pd.read_csv(f"{DRIVE}/type_predictions_labeled_v3.csv")
te_p1   = np.load(f"{DRIVE}/all_encoder_predictions.npz")['BiomedBERT_42']
y = test_df['label'].values

# ---- ensemble, cross-validated on the test features ----------------
# NOTE: this is NOT a held-out result. Each fold's classifier sees 80%
# of the test labels. Reported in the paper as a negative result only.
F = ['contradiction', 'neutral', 'entailment', 'phi']
X = StandardScaler().fit_transform(
        np.column_stack([te_p1] + [te_nli[c].values for c in F]))
p2 = cross_val_predict(
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                   max_depth=3, random_state=42),
        X, y, cv=5, method='predict_proba')[:, 1]

print("condition        macroF1   sens    spec    AUROC")
for n, s in [('Phase 1 (held out)', te_p1), ('Phase 2 (CV on test)', p2)]:
    pr = (s >= .5).astype(int)
    print(f"{n:22s} {f1_score(y, pr, average='macro'):.4f} "
          f"{recall_score(y, pr):.4f} {recall_score(y, pr, pos_label=0):.4f} "
          f"{roc_auc_score(y, s):.4f}")

print("\nper difficulty      n   Phase1   Phase2")
for d in ['easy', 'medium', 'hard']:
    m = (test_df['difficulty'] == d).values
    print(f"  {d:8s} {m.sum():>6d}   "
          f"{f1_score(y[m], (te_p1[m]>=.5).astype(int), average='macro'):.4f}   "
          f"{f1_score(y[m], (p2[m]>=.5).astype(int), average='macro'):.4f}")

# ---- severity ------------------------------------------------------
# R is MedHallu's difficulty label: benchmark metadata, not derivable
# from (q, a, k). The score is therefore an analysis of labelled data,
# not a deployable component.
h = np.where(y == 1)[0]
W = {0: .7, 1: .4, 2: .8, 3: 1.}
R = {'hard': 1., 'medium': .6, 'easy': .2}
sev = pd.DataFrame({'difficulty': test_df.loc[h, 'difficulty'].values,
                    'p1': te_p1[h],
                    'contra': te_nli['contradiction'].values[h]})
sev['Wt'] = [W[i] for i in tp['pred_tid'].values]
sev['severity'] = np.minimum(
    sev['Wt'] * (.6*sev['p1'] + .4*sev['contra']) *
    sev['difficulty'].map(R), 1.)
sev['band'] = pd.cut(sev['severity'], [-.01, .3, .7, 1.01],
                     labels=['LOW', 'MODERATE', 'HIGH'])
rho, pv = spearmanr(sev['severity'],
                    sev['difficulty'].map({'easy':0,'medium':1,'hard':2}))
print(f"\nseverity: rho = {rho:.4f} (p = {pv:.2e})")
print("  NOTE: difficulty is a factor inside S, so this correlation is")
print("        an internal-consistency check, not validation.")
print("  bands:", sev['band'].value_counts().to_dict())
print("  means:", {d: round(sev[sev.difficulty==d]['severity'].mean(), 4)
                   for d in ['easy','medium','hard']})

# ---- contradiction signal ------------------------------------------
ph = te_nli['contradiction'].values[y == 1]
pc = te_nli['contradiction'].values[y == 0]
sp = np.sqrt(((len(ph)-1)*ph.std(ddof=1)**2 +
              (len(pc)-1)*pc.std(ddof=1)**2) / (len(ph)+len(pc)-2))
d = (ph.mean() - pc.mean()) / sp
print(f"\ncontradiction: hallucinated {ph.mean():.4f} vs clean "
      f"{pc.mean():.4f}, Cohen's d = {d:.4f}, "
      f"p = {mannwhitneyu(ph, pc, alternative='greater').pvalue:.2e}")

sev.to_csv(f"{DRIVE}/severity_final.csv", index=False)
np.savez(f"{DRIVE}/figure_data_final.npz", y_true=y, y_score_p1=te_p1,
         y_score_p2=p2, difficulty=test_df['difficulty'].values,
         severity=sev['severity'].values,
         sev_band=sev['band'].astype(str).values,
         sev_difficulty=sev['difficulty'].values,
         phi_hall=ph, phi_clean=pc)
print("\nsaved figure_data_final.npz")

condition        macroF1   sens    spec    AUROC
Phase 1 (held out)     0.9630 0.9720 0.9540 0.9936
Phase 2 (CV on test)   0.9640 0.9650 0.9630 0.9924

per difficulty      n   Phase1   Phase2
  easy        548   0.9726   0.9635
  medium      636   0.9638   0.9670
  hard        816   0.9559   0.9620

severity: rho = 0.7934 (p = 2.22e-217)
  NOTE: difficulty is a factor inside S, so this correlation is
        an internal-consistency check, not validation.
  bands: {'LOW': 613, 'MODERATE': 383, 'HIGH': 4}
  means: {'easy': np.float64(0.0908), 'medium': np.float64(0.2676), 'hard': np.float64(0.4278)}

contradiction: hallucinated 0.3057 vs clean 0.0726, Cohen's d = 0.6666, p = 4.83e-54

saved figure_data_final.npz


In [ ]:
# ============================================================
# CELL 13 — Regenerate every figure in the paper
#
# IEEE style: serif face matching the body text, greyscale-safe
# palette, 600+ dpi. Column figures 3.5 in, page-wide 7.16 in.
# ============================================================
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, os, json, pandas as pd
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score)

FIGS = "/content/figures"; os.makedirs(FIGS, exist_ok=True)
d = np.load(f"{DRIVE}/figure_data_final.npz", allow_pickle=True)
seeds_df = pd.read_csv(f"{DRIVE}/all_encoder_seeds.csv")
short = json.load(open(f"{DRIVE}/shortcut_summary.json"))
cvcis = json.load(open(f"{DRIVE}/type_cv_cis.json"))

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "figure.dpi": 750, "savefig.dpi": 750, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02, "axes.spines.top": False,
    "axes.spines.right": False, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6})
GREY, DARK, OURS, GRID = "#B0B0B0", "#606060", "#1A1A1A", "#DDDDDD"
COL, PAGE = 3.5, 7.16
def save(f, n): f.savefig(f"{FIGS}/{n}", facecolor="white"); plt.close(f); print("wrote", n)

# ---- shortcut diagnostics (principal result) -----------------------
conds = ['full', 'answer_only', 'shuffled', 'evidence_only']
labs  = ['Full\n(q + evidence)', 'Answer\nonly', 'Shuffled\nevidence',
         'Evidence only\n(control)']
vals  = [short[c]['macro_f1'] for c in conds]
fig, ax = plt.subplots(figsize=(COL, 2.2))
b = ax.bar(range(4), vals, color=[OURS, DARK, DARK, GREY],
           edgecolor="black", linewidth=.5, width=.65)
for bar, v in zip(b, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+.015, f"{v:.3f}",
            ha="center", fontsize=6)
ax.axhline(0.5, color="black", ls=":", lw=.8)
ax.text(3.4, 0.52, "chance", fontsize=5.5, ha="right")
ax.set_xticks(range(4)); ax.set_xticklabels(labs, fontsize=6)
ax.set_ylabel("Macro F1"); ax.set_ylim(0, 1.08)
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
save(fig, "shortcut_chart.png")

# ---- encoder comparison, 5 seeds -----------------------------------
order = ['Bio_ClinicalBERT', 'RoBERTa', 'BioBERT', 'BiomedBERT']
piv = seeds_df.pivot(index='seed', columns='encoder', values='macro_f1')
means = [piv[e].mean() for e in order]
sds   = [piv[e].std(ddof=1) for e in order]
fig, ax = plt.subplots(figsize=(COL, 2.1))
b = ax.bar(range(4), means, yerr=sds, capsize=2.5,
           color=[GREY]*3+[OURS], edgecolor="black", linewidth=.5, width=.6,
           error_kw=dict(elinewidth=.6))
ax.set_xticks(range(4))
ax.set_xticklabels([e.replace('_','-') for e in order], rotation=20,
                   ha="right", fontsize=6)
ax.set_ylabel("Macro F1"); ax.set_ylim(0.93, 0.98)
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
save(fig, "comparison_chart.png")

# ---- multi-seed stability ------------------------------------------
sv = piv['BiomedBERT'].values
hv = seeds_df.pivot(index='seed', columns='encoder',
                    values='hard_f1')['BiomedBERT'].values
av = seeds_df.pivot(index='seed', columns='encoder',
                    values='auroc')['BiomedBERT'].values
fig, axes = plt.subplots(1, 3, figsize=(PAGE, 2.0))
for ax, (v, lab) in zip(axes, [(sv, "Macro F1"), (hv, "Hard-case F1"),
                               (av, "AUROC")]):
    m, s = v.mean(), v.std(ddof=1)
    ax.bar(range(5), v, color=GREY, edgecolor="black", linewidth=.5, width=.65)
    ax.axhline(m, color="black", ls="--", lw=.8)
    ax.axhspan(m-s, m+s, color="#EEEEEE", zorder=0)
    ax.set_xticks(range(5))
    ax.set_xticklabels([str(x) for x in piv.index], fontsize=6)
    ax.set_xlabel("Random seed"); ax.set_ylabel(lab)
    ax.set_ylim(max(0, v.min()-4*s), min(1.002, v.max()+3*s))
    ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
    ax.set_title(f"{lab}: {m:.4f} $\\pm$ {s:.4f}", fontsize=7)
fig.tight_layout(); save(fig, "multiseed_chart.png")

# ---- type classification, cross-validated --------------------------
types = ["Misinterpretation", "Mechanism\nmisattribution",
         "Incomplete\ninformation", "Evidence\nfabrication"]
f1s = [0.8309, 0.5901, 0.5012, 0.0400]
los = [cvcis["Misinterpretation"][0], cvcis["Mechanism misattr."][0],
       cvcis["Incomplete info"][0], cvcis["Evidence fabric."][0]]
his = [cvcis["Misinterpretation"][1], cvcis["Mechanism misattr."][1],
       cvcis["Incomplete info"][1], cvcis["Evidence fabric."][1]]
ns  = [6848, 1096, 1010, 46]
fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.2))
y = np.arange(4)[::-1]
ax = axes[0]
err = np.array([np.array(f1s)-np.array(los), np.array(his)-np.array(f1s)])
ax.barh(y, f1s, xerr=err, capsize=2.5, color=GREY, edgecolor="black",
        linewidth=.5, height=.55, error_kw=dict(elinewidth=.6))
for yi, v in zip(y, f1s):
    ax.text(v+0.03, yi, f"{v:.2f}", va="center", fontsize=6)
ax.set_yticks(y); ax.set_yticklabels(types, fontsize=6)
ax.set_xlabel("F1 score (5-fold CV, 95% CI)"); ax.set_xlim(0, 1.0)
ax.xaxis.grid(True, color=GRID); ax.set_axisbelow(True)
ax = axes[1]
ax.barh(y, ns, color=DARK, edgecolor="black", linewidth=.5, height=.55)
for yi, n in zip(y, ns):
    ax.text(n+90, yi, str(n), va="center", fontsize=6)
ax.set_yticks(y); ax.set_yticklabels([])
ax.set_xlabel("Training instances"); ax.set_xlim(0, 8000)
ax.xaxis.grid(True, color=GRID); ax.set_axisbelow(True)
fig.tight_layout(); save(fig, "type_chart.png")

# ---- severity -------------------------------------------------------
sev, sd = d['severity'], d['sev_difficulty']
fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.2))
bands = ["LOW", "MODERATE", "HIGH"]
cnt = [int((d['sev_band'] == b).sum()) for b in bands]
ax = axes[0]
bb = ax.bar([b.capitalize() for b in bands], cnt, color=GREY,
            edgecolor="black", linewidth=.5, width=.6)
for bar, c in zip(bb, cnt):
    ax.text(bar.get_x()+bar.get_width()/2, c+8, str(c), ha="center", fontsize=6)
ax.set_ylabel("Count"); ax.set_title("Severity band distribution", fontsize=7)
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
ax = axes[1]
bp = ax.boxplot([sev[sd == t] for t in ['easy','medium','hard']],
                tick_labels=["Easy","Medium","Hard"], widths=.5,
                patch_artist=True,
                flierprops=dict(marker="o", markersize=1.5,
                                markerfacecolor="none", markeredgewidth=.4))
for p in bp["boxes"]:
    p.set_facecolor("#EEEEEE"); p.set_edgecolor("black"); p.set_linewidth(.6)
for el in ["whiskers","caps","medians"]:
    for ln in bp[el]: ln.set_color("black"); ln.set_linewidth(.6)
ax.set_ylabel("Severity score $S$")
ax.set_title("Severity by difficulty (internal consistency)", fontsize=7)
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
fig.tight_layout(); save(fig, "severity_chart.png")

# ---- contradiction signal -------------------------------------------
ph, pc = d['phi_hall'], d['phi_clean']
fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.2))
ax = axes[0]; bins = np.linspace(0, 1, 26)
ax.hist(pc, bins=bins, density=True, color="#DDDDDD", edgecolor="black",
        linewidth=.4, label="Non-hallucinated")
ax.hist(ph, bins=bins, density=True, color=DARK, edgecolor="black",
        linewidth=.4, alpha=.75, label="Hallucinated")
ax.set_xlabel("Contradiction score"); ax.set_ylabel("Density")
ax.legend(frameon=False); ax.set_title("Score distribution", fontsize=7)
ax = axes[1]
bp = ax.boxplot([pc, ph], tick_labels=["Non-hallucinated","Hallucinated"],
                widths=.45, patch_artist=True,
                flierprops=dict(marker="o", markersize=1.5,
                                markerfacecolor="none", markeredgewidth=.4))
for p in bp["boxes"]:
    p.set_facecolor("#EEEEEE"); p.set_edgecolor("black"); p.set_linewidth(.6)
for el in ["whiskers","caps","medians"]:
    for ln in bp[el]: ln.set_color("black"); ln.set_linewidth(.6)
ax.set_ylabel("Contradiction score")
ax.set_title("Separation by class", fontsize=7)
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
fig.tight_layout(); save(fig, "contradiction_chart.png")

# ---- ROC / PR --------------------------------------------------------
yv, p1, p2 = d['y_true'], d['y_score_p1'], d['y_score_p2']
fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.4))
ax = axes[0]
for s, lab, st in [(p1, "Phase 1 (held out)", "-"),
                   (p2, "Phase 2 (CV on test)", "--")]:
    fpr, tpr, _ = roc_curve(yv, s)
    ax.plot(fpr, tpr, st, color="black",
            label=f"{lab} (AUC {auc(fpr, tpr):.4f})")
ax.plot([0,1],[0,1], ":", color=GREY, label="Random")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(frameon=False, loc="lower right")
ax.set_title("Receiver operating characteristic", fontsize=7)
ax = axes[1]
for s, lab, st in [(p1, "Phase 1 (held out)", "-"),
                   (p2, "Phase 2 (CV on test)", "--")]:
    pr, rc, _ = precision_recall_curve(yv, s)
    ax.plot(rc, pr, st, color="black",
            label=f"{lab} (AUPRC {average_precision_score(yv, s):.4f})")
ax.axhline(.5, ls=":", color=GREY, label="Random")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_ylim(.45, 1.02)
ax.legend(frameon=False, loc="lower left")
ax.set_title("Precision-recall", fontsize=7)
fig.tight_layout(); save(fig, "roc_pr_curves.png")

from PIL import Image
print("\nfigure sizes:")
for f in sorted(os.listdir(FIGS)):
    print(f"  {f:28s} {Image.open(f'{FIGS}/{f}').size}")
print("\nDownload with:  from google.colab import files")
print("                for f in os.listdir(FIGS): files.download(f'{FIGS}/{f}')")

wrote shortcut_chart.png
wrote comparison_chart.png
wrote multiseed_chart.png
wrote type_chart.png
wrote severity_chart.png
wrote contradiction_chart.png
wrote roc_pr_curves.png

figure sizes:
  comparison_chart.png         (2421, 1586)
  contradiction_chart.png      (5219, 1500)
  multiseed_chart.png          (5220, 1350)
  roc_pr_curves.png            (5220, 1649)
  severity_chart.png           (5220, 1500)
  shortcut_chart.png           (2374, 1506)
  type_chart.png               (5220, 1500)

Download with:  from google.colab import files
                for f in os.listdir(FIGS): files.download(f'{FIGS}/{f}')


In [ ]:
from google.colab import files
import os
for f in sorted(os.listdir("/content/figures")):
    files.download(f"/content/figures/{f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# ============================================================
# CELL 14 — Reviewer-response experiments
#
#   A9  answer-only surface baselines (length, style, TF-IDF)   CPU, ~3 min
#   A1  test-time ablation of the saved full-input model        GPU, ~5 min
#   A2  training-time evidence ablation across all five seeds   GPU, ~2 h
#
# Run order: Cell 1 (mount), then this cell. Nothing else needed.
# Every unit caches to Drive under medhallu_v2/review_v14/, so if
# Colab disconnects, just re-run Cell 1 and this cell: finished
# units are skipped and only the interrupted run repeats.
#
# At the end it prints a block starting "PASTE THIS BACK". Send
# that block back to Claude.
# ============================================================
import os, json, re, time
import numpy as np, pandas as pd, torch
from scipy import stats
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

DRIVE = "/content/drive/MyDrive/medhallu_v2"
OUT = f"{DRIVE}/review_v14"
assert os.path.exists(f"{DRIVE}/train_df.csv"), "Run Cell 1 (Drive mount) first"
os.makedirs(OUT, exist_ok=True)

BIOMEDBERT = ("microsoft/BiomedNLP-BiomedBERT-base-"
              "uncased-abstract-fulltext")
SEEDS = [42, 123, 456, 789, 1234]

train_df = pd.read_csv(f"{DRIVE}/train_df.csv")
test_df = pd.read_csv(f"{DRIVE}/test_df.csv")
for d in (train_df, test_df):
    n_nan = d[['question', 'knowledge', 'answer']].isna().sum().sum()
    if n_nan:
        print(f"note: {n_nan} empty text fields filled with ''")
    for c in ['question', 'knowledge', 'answer']:
        d[c] = d[c].fillna('').astype(str)

y_tr = train_df['label'].values
y_te = test_df['label'].values
hard = (test_df['difficulty'] == 'hard').values


def score(p, y=y_te, mask=hard):
    """Macro F1, hard-tier macro F1 and AUROC from P(hallucinated)."""
    pred = (np.asarray(p) >= 0.5).astype(int)
    out = {'macro_f1': round(float(f1_score(y, pred, average='macro')), 4),
           'auroc': round(float(roc_auc_score(y, p)), 4)}
    if mask is not None:
        out['hard_f1'] = round(float(f1_score(y[mask], pred[mask],
                                              average='macro')), 4)
    return out


def shuffle_df(df, rng):
    """Give every row the question and passage of a different row."""
    d = df.copy().reset_index(drop=True)
    n = len(d)
    perm = rng.permutation(n)
    while True:
        same = perm == np.arange(n)
        if not same.any():
            break
        perm[same] = (perm[same] + 1) % n
    d['knowledge'] = d['knowledge'].values[perm]
    d['question'] = d['question'].values[perm]
    return d


# =====================================================================
# A9 — answer-only surface baselines.  No GPU, no evidence (except the
#      overlap feature, which is labelled as evidence-using).
# =====================================================================
HEDGE = (r"\b(may|might|could|possibly|potentially|suggest(s|ed|ing)?|"
         r"likely|appear(s|ed)?|seem(s|ed)?|unclear)\b")
NEG = r"\b(not|no|neither|nor|without|never)\b|n't"
CERT = r"\b(clearly|definitely|significantly|strongly|always|demonstrates?|proves?|confirms?|establish(es|ed)?)\b"
TOKEN = re.compile(r"[a-z0-9]+")
STOP = set("the a an of in on and or to for with is are was were be been by as at "
           "that this from it its which these those than into".split())


def style_features(df):
    a = df['answer']
    low = a.str.lower()
    n_words = a.str.split().str.len().fillna(0)
    f = pd.DataFrame({
        'log_words': np.log1p(n_words),
        'log_chars': np.log1p(a.str.len()),
        'n_sents': a.str.count(r"[.!?](\s|$)"),
        'avg_word_len': a.str.replace(r"\s+", "", regex=True).str.len()
                        / n_words.clip(lower=1),
        'digits': low.str.count(r"\d"),
        'commas': a.str.count(","),
        'parens': a.str.count(r"\("),
        'hedges': low.str.count(HEDGE),
        'negations': low.str.count(NEG),
        'certainty': low.str.count(CERT),
        'starts_yes_no': low.str.match(r"^\s*(yes|no)\b").astype(int),
    })
    return f.fillna(0)


def overlap_feature(df):
    """Share of the answer's content words that appear in the passage.
    This one DOES read the evidence, so it is reported separately."""
    vals = []
    for a, k in zip(df['answer'], df['knowledge']):
        aw = {w for w in TOKEN.findall(a.lower()) if w not in STOP}
        kw = set(TOKEN.findall(k.lower()))
        vals.append(len(aw & kw) / len(aw) if aw else 0.0)
    return np.array(vals).reshape(-1, 1)


def load_halueval():
    from datasets import load_dataset
    ds = load_dataset("pminervini/HaluEval", "qa")['data'].select(range(2000))
    rows = []
    for ex in ds:
        base = {'question': ex['question'], 'knowledge': ex['knowledge']}
        rows.append({**base, 'answer': ex['right_answer'], 'label': 0})
        rows.append({**base, 'answer': ex['hallucinated_answer'], 'label': 1})
    return pd.DataFrame(rows)


def a9_baselines():
    path = f"{OUT}/a9_baselines.json"
    if os.path.exists(path):
        print("A9: cached")
        return json.load(open(path))
    t0 = time.time()
    res = {}

    # descriptive length statistics
    desc = {}
    for name, d in [('train', train_df), ('test', test_df)]:
        w = d['answer'].str.split().str.len()
        desc[name] = {'correct_median_words': float(w[d.label == 0].median()),
                      'halluc_median_words': float(w[d.label == 1].median()),
                      'correct_mean_words': round(float(w[d.label == 0].mean()), 1),
                      'halluc_mean_words': round(float(w[d.label == 1].mean()), 1)}
    res['length_stats'] = desc

    lr = lambda: LogisticRegression(max_iter=5000, C=1.0)

    # 1) length only
    Xtr, Xte = style_features(train_df), style_features(test_df)
    m = make_pipeline(StandardScaler(), lr()).fit(Xtr[['log_words']], y_tr)
    res['length_only'] = score(m.predict_proba(Xte[['log_words']])[:, 1])

    # 2) all hand-crafted style features
    m_style = make_pipeline(StandardScaler(), lr()).fit(Xtr, y_tr)
    res['style_features'] = score(m_style.predict_proba(Xte)[:, 1])
    coefs = m_style[-1].coef_[0]
    res['style_coefficients'] = {c: round(float(v), 3)
                                 for c, v in sorted(zip(Xtr.columns, coefs),
                                                    key=lambda t: -abs(t[1]))}

    # 3) TF-IDF bag of words on the answer alone
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                          max_features=50000, lowercase=True)
    Atr = vec.fit_transform(train_df['answer'])
    Ate = vec.transform(test_df['answer'])
    m_tfidf = LogisticRegression(max_iter=5000, C=1.0).fit(Atr, y_tr)
    p_tfidf = m_tfidf.predict_proba(Ate)[:, 1]
    res['tfidf_answer_only'] = score(p_tfidf)
    np.save(f"{OUT}/a9_tfidf_test_probs.npy", p_tfidf)
    names = np.array(vec.get_feature_names_out())
    order = np.argsort(m_tfidf.coef_[0])
    res['tfidf_top_hallucinated'] = names[order[::-1][:30]].tolist()
    res['tfidf_top_correct'] = names[order[:30]].tolist()

    # 4) lexical overlap with the passage (uses evidence)
    Otr, Ote = overlap_feature(train_df), overlap_feature(test_df)
    m = LogisticRegression(max_iter=5000).fit(Otr, y_tr)
    res['overlap_with_evidence'] = score(m.predict_proba(Ote)[:, 1])
    res['overlap_means'] = {
        'correct': round(float(Ote[y_te == 0].mean()), 4),
        'hallucinated': round(float(Ote[y_te == 1].mean()), 4)}

    # 5) does the TF-IDF shortcut transfer to HaluEval?
    try:
        ext = load_halueval()
        pe = m_tfidf.predict_proba(vec.transform(ext['answer']))[:, 1]
        res['tfidf_halueval'] = score(pe, y=ext['label'].values, mask=None)
    except Exception as e:
        res['tfidf_halueval'] = {'error': str(e)}

    json.dump(res, open(path, "w"), indent=2)
    print(f"A9 done in {time.time()-t0:.0f}s")
    return res


# =====================================================================
# A1 — test-time ablation of the SAVED full-input model (biomedbert_v2).
#      No training. Answers: does the model that was trained WITH
#      evidence use the evidence when it predicts?
# =====================================================================
@torch.no_grad()
def predict_saved(model, tok, df, batch=64):
    seg1 = [f"{q} {k}" if (q or k) else "" for q, k in
            zip(df['question'], df['knowledge'])]
    seg2 = list(df['answer'])
    probs = []
    for i in range(0, len(df), batch):
        enc = tok(seg1[i:i+batch], seg2[i:i+batch],
                  truncation='longest_first', max_length=512,
                  padding=True, return_tensors='pt').to('cuda')
        probs.append(torch.softmax(model(**enc).logits.float(), -1)
                     .cpu().numpy()[:, 1])
    return np.concatenate(probs)


def a1_test_time():
    path = f"{OUT}/a1_test_time.json"
    if os.path.exists(path):
        print("A1: cached")
        return json.load(open(path))
    assert torch.cuda.is_available(), "A1 needs a GPU runtime"
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(f"{DRIVE}/biomedbert_v2_tok")
    model = AutoModelForSequenceClassification.from_pretrained(
        f"{DRIVE}/biomedbert_v2").cuda().eval()

    variants = {
        'full': test_df,
        'evidence_removed': test_df.assign(question='', knowledge=''),
        'evidence_shuffled': shuffle_df(test_df, np.random.default_rng(0)),
        'answer_removed': test_df.assign(answer=''),
    }
    res, P = {}, {}
    for name, d in variants.items():
        P[name] = predict_saved(model, tok, d)
        res[name] = score(P[name])
        print(f"  A1 {name:18s} {res[name]}", flush=True)
    base = (P['full'] >= .5)
    for name in ['evidence_removed', 'evidence_shuffled']:
        res[name]['pred_changed_vs_full_pct'] = round(
            float(((P[name] >= .5) != base).mean() * 100), 2)
        res[name]['mean_abs_prob_change'] = round(
            float(np.abs(P[name] - P['full']).mean()), 4)
    np.savez(f"{OUT}/a1_test_time_probs.npz", **P)
    json.dump(res, open(path, "w"), indent=2)
    del model; torch.cuda.empty_cache()
    print(f"A1 done in {time.time()-t0:.0f}s")
    return res


# =====================================================================
# A2 — training-time ablation, five seeds.
#   full        : the five BiomedBERT runs already in
#                 all_encoder_predictions.npz (same recipe, same seeds),
#                 so the paper has ONE full-input number everywhere.
#   answer_only : seed 42 from Cell 8 cache, seeds 123..1234 trained here
#   shuffled    : seed 42 from Cell 8 cache, seeds 123..1234 trained here
#   evidence_only (control): single run from Cell 8, reported as is.
# Same recipe as Cell 8 ("[EMPTY]" placeholder for removed segments).
# Padding is dynamic instead of fixed 512: masked positions do not
# affect the output, it only makes the short answer-only runs faster.
# =====================================================================
class AblDS(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.a = [(f"{q} {k}".strip() or "[EMPTY]")
                  for q, k in zip(df['question'], df['knowledge'])]
        self.b = [(str(x).strip() or "[EMPTY]") for x in df['answer']]
        self.y = df['label'].astype(int).values
        self.tok = tok
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        enc = self.tok(self.a[i], self.b[i], truncation='longest_first',
                       max_length=512)
        enc['labels'] = int(self.y[i])
        return enc


def train_ablation(cond, seed):
    path = f"{OUT}/a2_{cond}_{seed}.json"
    if os.path.exists(path):
        print(f"A2 {cond} seed {seed}: cached")
        return json.load(open(path))
    assert torch.cuda.is_available(), "A2 needs a GPU runtime"
    t0 = time.time()
    if cond == 'answer_only':
        tr = train_df.assign(question='', knowledge='')
        te = test_df.assign(question='', knowledge='')
    elif cond == 'shuffled':
        rng = np.random.default_rng(seed)
        tr, te = shuffle_df(train_df, rng), shuffle_df(test_df, rng)
    else:
        raise ValueError(cond)

    tok = AutoTokenizer.from_pretrained(BIOMEDBERT)
    model = AutoModelForSequenceClassification.from_pretrained(
        BIOMEDBERT, num_labels=2)
    args = TrainingArguments(
        output_dir=f"/content/a2_{cond}_{seed}", num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="no", save_strategy="no", warmup_steps=500,
        logging_steps=1000, fp16=True, seed=seed, report_to="none")
    t = Trainer(model=model, args=args, train_dataset=AblDS(tr, tok),
                data_collator=DataCollatorWithPadding(tok))
    t.train()
    logits = t.predict(AblDS(te, tok)).predictions
    p = torch.softmax(torch.tensor(logits), -1).numpy()[:, 1]
    res = {'condition': cond, 'seed': seed, **score(p),
           'minutes': round((time.time() - t0) / 60, 1)}
    np.save(f"{OUT}/a2_{cond}_{seed}_probs.npy", p)
    json.dump(res, open(path, "w"), indent=2)
    del model, t; torch.cuda.empty_cache()
    print(f"A2 {cond} seed {seed}: {res}", flush=True)
    return res


def a2_multiseed():
    rows = []
    enc = np.load(f"{DRIVE}/all_encoder_predictions.npz")
    for s in SEEDS:
        rows.append({'condition': 'full', 'seed': s,
                     **score(enc[f"BiomedBERT_{s}"])})
    for cond in ['answer_only', 'shuffled']:
        for s in SEEDS:
            cached = f"{DRIVE}/shortcut_{cond}.json"
            if s == 42 and os.path.exists(cached):
                r = json.load(open(cached))
                rows.append({'condition': cond, 'seed': 42,
                             'macro_f1': r['macro_f1'],
                             'hard_f1': r['hard_f1'], 'auroc': r['auroc']})
            else:
                r = train_ablation(cond, s)
                rows.append({k: r[k] for k in
                             ['condition', 'seed', 'macro_f1', 'hard_f1',
                              'auroc']})
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT}/a2_multiseed.csv", index=False)

    tcrit = stats.t.ppf(.975, len(SEEDS) - 1)
    summ = {}
    for cond in ['full', 'answer_only', 'shuffled']:
        d = df[df.condition == cond].set_index('seed').loc[SEEDS]
        summ[cond] = {}
        for m in ['macro_f1', 'hard_f1', 'auroc']:
            v = d[m].values
            mu, sd = v.mean(), v.std(ddof=1)
            h = tcrit * sd / np.sqrt(len(v))
            summ[cond][m] = {'mean': round(mu, 4), 'sd': round(sd, 4),
                             'ci95': [round(mu - h, 4), round(mu + h, 4)]}
    full = df[df.condition == 'full'].set_index('seed').loc[SEEDS]['macro_f1'].values
    paired = {}
    for cond in ['answer_only', 'shuffled']:
        v = df[df.condition == cond].set_index('seed').loc[SEEDS]['macro_f1'].values
        diff = full - v
        h = tcrit * diff.std(ddof=1) / np.sqrt(len(diff))
        tstat, p = stats.ttest_rel(full, v)
        paired[cond] = {
            'mean_drop': round(float(diff.mean()), 4),
            'ci95': [round(float(diff.mean() - h), 4),
                     round(float(diff.mean() + h), 4)],
            'p': round(float(p), 4),
            'pct_of_full': round(float((v / full).mean() * 100), 1),
            'pct_of_above_chance': round(float(((v - .5) / (full - .5)).mean()
                                               * 100), 1)}
    ctrl = json.load(open(f"{DRIVE}/shortcut_evidence_only.json"))
    out = {'per_seed': rows, 'summary': summ, 'paired_vs_full': paired,
           'evidence_only_control_seed42': ctrl}
    json.dump(out, open(f"{OUT}/a2_summary.json", "w"), indent=2)
    return out


# =====================================================================
# RUN — fastest first, so useful results arrive early.
# =====================================================================
r9 = a9_baselines()
print(json.dumps({k: r9[k] for k in ['length_only', 'style_features',
                                     'tfidf_answer_only']}, indent=1))
r1 = a1_test_time()
r2 = a2_multiseed()

print("\n" + "=" * 64)
print("PASTE THIS BACK")
print("=" * 64)
print(json.dumps({'A9': r9, 'A1': r1,
                  'A2_summary': r2['summary'],
                  'A2_paired_vs_full': r2['paired_vs_full'],
                  'A2_per_seed': r2['per_seed'],
                  'A2_control': r2['evidence_only_control_seed42']},
                 indent=1))

README.md:   0%|          | 0.00/2.88k [00:00<?, ?B/s]

qa/data-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.75MB            

qa/data-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

A9 done in 12s
{
 "length_only": {
  "macro_f1": 0.717,
  "auroc": 0.7396,
  "hard_f1": 0.7341
 },
 "style_features": {
  "macro_f1": 0.7929,
  "auroc": 0.8748,
  "hard_f1": 0.7959
 },
 "tfidf_answer_only": {
  "macro_f1": 0.9073,
  "auroc": 0.9744,
  "hard_f1": 0.9018
 }
}


...

A2 shuffled seed 789: {'condition': 'shuffled', 'seed': 789, 'macro_f1': 0.9535, 'auroc': 0.9931, 'hard_f1': 0.9546, 'minutes': 5.5}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

Step,Training Loss
1000,0.278129
2000,0.121188
3000,0.065164


A2 shuffled seed 1234: {'condition': 'shuffled', 'seed': 1234, 'macro_f1': 0.9485, 'auroc': 0.9926, 'hard_f1': 0.9423, 'minutes': 5.4}

PASTE THIS BACK
{
 "A9": {
  "length_stats": {
   "train": {
    "correct_median_words": 34.0,
    "halluc_median_words": 22.0,
    "correct_mean_words": 37.5,
    "halluc_mean_words": 27.7
   },
   "test": {
    "correct_median_words": 37.0,
    "halluc_median_words": 23.0,
    "correct_mean_words": 39.7,
    "halluc_mean_words": 27.8
   }
  },
  "length_only": {
   "macro_f1": 0.717,
   "auroc": 0.7396,
   "hard_f1": 0.7341
  },
  "style_features": {
   "macro_f1": 0.7929,
   "auroc": 0.8748,
   "hard_f1": 0.7959
  },
  "style_coefficients": {
   "log_words": -1.83,
   "log_chars": 1.398,
   "hedges": -0.896,
   "n_sents": -0.626,
   "parens": 0.618,
   "avg_word_len": 0.404,
   "certainty": 0.38,
   "commas": 0.264,
   "digits": 0.215,
   "negations": -0.036,
   "starts_yes_no": -0.023
  },
  "tfidf_answer_only": {
   "macro_f1": 0.9073,
   "auroc":

In [4]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
P = np.load(f"{DRIVE}/review_v14/a1_test_time_probs.npz")
t = pd.read_csv(f"{DRIVE}/test_df.csv"); y = t['label'].values
assert (y[0::2] == 0).all() and (y[1::2] == 1).all()
for k in P.files:
    p = P[k]; pred = p >= .5
    best = max(f1_score(y, p >= th, average='macro') for th in np.linspace(.01, .99, 99))
    pair = (p[1::2] > p[0::2]).mean()
    print(f"{k:18s} flagged: correct {pred[y==0].mean():.3f} halluc {pred[y==1].mean():.3f} | "
          f"mean P: correct {p[y==0].mean():.3f} halluc {p[y==1].mean():.3f} | "
          f"best-threshold F1 {best:.4f} | pairwise acc {pair:.4f}")

full               flagged: correct 0.021 halluc 0.964 | mean P: correct 0.024 halluc 0.965 | best-threshold F1 0.9730 | pairwise acc 0.9930
evidence_removed   flagged: correct 0.247 halluc 0.978 | mean P: correct 0.334 halluc 0.936 | best-threshold F1 0.9240 | pairwise acc 0.9780
evidence_shuffled  flagged: correct 0.599 halluc 1.000 | mean P: correct 0.602 halluc 0.999 | best-threshold F1 0.8696 | pairwise acc 0.9850
answer_removed     flagged: correct 0.858 halluc 0.858 | mean P: correct 0.810 halluc 0.810 | best-threshold F1 0.4999 | pairwise acc 0.0000


In [6]:
# ============================================================
# CELL 15 — New Figure 3 (evidence ablation, five seeds)
#
# Replaces shortcut_chart.png. Page-wide, two panels:
#   (a) all four conditions on 0-1, control annotated
#   (b) the three answer-bearing conditions zoomed, seeds as dots
# Run after Cell 14. Needs only the files Cell 14 wrote.
# ============================================================
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, json, os

DRIVE = "/content/drive/MyDrive/medhallu_v2"
FIGS = "/content/figures"; os.makedirs(FIGS, exist_ok=True)
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 8, "axes.labelsize": 8, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "savefig.dpi": 750, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02, "axes.spines.top": False,
    "axes.spines.right": False, "axes.linewidth": 0.6})
GREY, DARK, OURS, GRID = "#B0B0B0", "#606060", "#1A1A1A", "#DDDDDD"

df = pd.read_csv(f"{DRIVE}/review_v14/a2_multiseed.csv")
ctrl = json.load(open(f"{DRIVE}/shortcut_evidence_only.json"))['macro_f1']
conds = ['full', 'answer_only', 'shuffled']
labs = ['Full\n(q + evidence)', 'Answer\nonly', 'Shuffled\nevidence']
vals = {c: df[df.condition == c]['macro_f1'].values for c in conds}
means = [vals[c].mean() for c in conds]
sds = [vals[c].std(ddof=1) for c in conds]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.16, 2.3),
                             gridspec_kw={"width_ratios": [1.15, 1]})

# (a) all four conditions
x = np.arange(4)
h = means + [ctrl]
a1.bar(x, h, yerr=sds + [0], capsize=2.5, width=.62,
       color=[OURS, DARK, DARK, GREY], edgecolor="black", linewidth=.5,
       error_kw=dict(elinewidth=.6))
for xi, v in zip(x, h):
    a1.text(xi, v + .03, f"{v:.3f}", ha="center", fontsize=6)
a1.text(3, ctrl / 2, "single-class\nprediction", ha="center", va="center",
        fontsize=5.5)
a1.set_xticks(x); a1.set_xticklabels(labs + ['Evidence only\n(control)'],
                                     fontsize=6)
a1.set_ylabel("Macro F1"); a1.set_ylim(0, 1.08)
a1.yaxis.grid(True, color=GRID); a1.set_axisbelow(True)
a1.set_title("(a) All conditions", fontsize=7)

# (b) zoom, seeds as dots
rng = np.random.default_rng(0)
for i, c in enumerate(conds):
    a2.bar(i, means[i], width=.55, color="white", edgecolor="black",
           linewidth=.6)
    a2.errorbar(i, means[i], yerr=sds[i], color="black", capsize=3,
                elinewidth=.7)
    jit = rng.uniform(-.12, .12, len(vals[c]))
    a2.scatter(i + jit, vals[c], s=9, color=OURS if c == 'full' else DARK,
               zorder=3, linewidths=0)
    a2.text(i, max(vals[c].max(), means[i] + sds[i]) + .0012,
            f"{means[i]:.4f}", ha="center",
            fontsize=6)
a2.set_xticks(range(3)); a2.set_xticklabels(labs, fontsize=6)
a2.set_ylabel("Macro F1"); a2.set_ylim(.94, .98)
a2.yaxis.grid(True, color=GRID); a2.set_axisbelow(True)
a2.set_title("(b) Answer-bearing conditions, five seeds (y axis from 0.94)",
             fontsize=7)

fig.tight_layout()
fig.savefig(f"{FIGS}/shortcut_chart.png", facecolor="white")
plt.close(fig)
print("wrote shortcut_chart.png")
from google.colab import files
files.download(f"{FIGS}/shortcut_chart.png")

wrote shortcut_chart.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# ============================================================
# CELL 16 — New experiments for the revision.  STANDALONE.
#
# Needs only: Cell 1 (Drive mount) run first, and an A100 runtime.
# Does NOT need any earlier cell to have been run in this session.
#
#   P0  fact checks (CPU, ~2 min)
#       - are MedHallu's correct answers PubMedQA's long answers?
#       - how the random / zero-shot NLI baselines were defined
#       - HaluEval answer lengths (why the TF-IDF cues reverse)
#       - AUROC of the contradiction score (rank effect size)
#   P1  length-matched subset (CPU, seconds; uses saved predictions)
#   P2  answer-only detector, retrained once and SAVED (GPU, ~3 min)
#   P3  style-controlled test set: an LLM rewrites answers so both
#       classes share one register, then every model is re-scored
#       (GPU, ~30-40 min)
#   P4  the same ablation inside HaluEval QA (GPU, ~25 min)
#
# Everything caches to Drive under medhallu_v2/review_v16/. If Colab
# disconnects: run Cell 1, then this cell again. Finished parts are
# skipped; an interrupted rewrite resumes where it stopped.
#
# At the end it prints "PASTE THIS BACK". Send that block to Claude,
# plus the file review_v16/p3_manual_check.csv (50 rewrites for you
# to read, see the instructions printed at the end).
# ============================================================
import os, json, re, time, gc
import numpy as np, pandas as pd, torch
from scipy import stats
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForCausalLM, TrainingArguments, Trainer,
                          DataCollatorWithPadding)
from datasets import load_dataset

DRIVE = "/content/drive/MyDrive/medhallu_v2"
OUT = f"{DRIVE}/review_v16"
assert os.path.exists(f"{DRIVE}/train_df.csv"), "Run Cell 1 (Drive mount) first"
os.makedirs(OUT, exist_ok=True)
BIOMEDBERT = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
NLI_MODEL = "cross-encoder/nli-deberta-v3-large"
REWRITER = "Qwen/Qwen2.5-7B-Instruct"
RESULTS = {}


def load_split(name):
    d = pd.read_csv(f"{DRIVE}/{name}_df.csv")
    for c in ['question', 'knowledge', 'answer']:
        d[c] = d[c].fillna('').astype(str)
    return d


train_df, test_df = load_split("train"), load_split("test")
y_te = test_df['label'].values
assert (y_te[0::2] == 0).all() and (y_te[1::2] == 1).all(), \
    "expected rows to alternate correct / hallucinated per item"


def cached(name, fn):
    path = f"{OUT}/{name}.json"
    if os.path.exists(path):
        print(f"[{name}] cached")
        return json.load(open(path))
    t0 = time.time()
    r = fn()
    json.dump(r, open(path, "w"), indent=1)
    print(f"[{name}] done in {time.time()-t0:.0f}s")
    return r


def score(p, y):
    p = np.asarray(p, float); y = np.asarray(y)
    pred = (p >= .5).astype(int)
    out = {'macro_f1': round(float(f1_score(y, pred, average='macro')), 4),
           'auroc': round(float(roc_auc_score(y, p)), 4)}
    if len(y) % 2 == 0 and (y[0::2] == 0).all() and (y[1::2] == 1).all():
        out['pairwise_acc'] = round(float((p[1::2] > p[0::2]).mean()), 4)
        out['correct_flagged'] = round(float(pred[y == 0].mean()), 4)
    return out


def words(s):
    return len(str(s).split())


# =====================================================================
# P0 — fact checks
# =====================================================================
def p0():
    r = {}
    # (a) provenance of MedHallu's correct answers
    pq = load_dataset("qiaojin/PubMedQA", "pqa_labeled")['train']
    la = {q.strip(): a.strip() for q, a in zip(pq['question'], pq['long_answer'])}
    corr = test_df[test_df.label == 0]
    hit = [la.get(q.strip()) for q in corr['question']]
    found = [h is not None for h in hit]
    exact = [h is not None and h == a.strip() for h, a in zip(hit, corr['answer'])]
    r['provenance'] = {
        'questions_found_in_pubmedqa': int(sum(found)),
        'correct_answer_equals_long_answer': int(sum(exact)),
        'n': int(len(corr)),
        'example_medhallu': corr['answer'].iloc[0][:300],
        'example_pubmedqa_long_answer': (hit[0] or '')[:300]}
    # (b) baseline definitions
    nli = pd.read_csv(f"{DRIVE}/nli_test.csv")
    cand = {'phi_gt_0': (nli['phi'] > 0).astype(int),
            'argmax_is_contradiction':
                (nli[['contradiction', 'entailment', 'neutral']].values.argmax(1) == 0).astype(int),
            'not_entailment_argmax':
                (nli[['contradiction', 'entailment', 'neutral']].values.argmax(1) != 1).astype(int)}
    r['zero_shot_nli_candidates'] = {k: round(float(f1_score(y_te, v, average='macro')), 4)
                                     for k, v in cand.items()}
    r['contradiction_score_auroc'] = round(float(roc_auc_score(y_te, nli['contradiction'])), 4)
    for f in ['results_v3_SUPERSEDED.json', 'results_v3.json', 'results_v2.json']:
        pth = f"{DRIVE}/{f}"
        if os.path.exists(pth):
            j = json.load(open(pth))
            r['stored_baselines'] = j.get('baselines_v2', j.get('baselines', 'not found'))
            r['stored_baselines_file'] = f
            break
    # (c) HaluEval lengths
    he = load_dataset("pminervini/HaluEval", "qa")['data']
    rl = np.array([words(a) for a in he['right_answer']])
    hl = np.array([words(a) for a in he['hallucinated_answer']])
    r['halueval_lengths'] = {'right_median_words': float(np.median(rl)),
                             'halluc_median_words': float(np.median(hl)),
                             'halluc_longer_pct': round(float((hl > rl).mean() * 100), 1)}
    return r


# =====================================================================
# P1 — length-matched subset, from predictions already on Drive
# =====================================================================
def p1():
    lc = test_df['answer'].iloc[0::2].map(words).values
    lh = test_df['answer'].iloc[1::2].map(words).values
    diff = np.abs(lc - lh)
    enc = np.load(f"{DRIVE}/all_encoder_predictions.npz")
    rv = f"{DRIVE}/review_v14"
    models = {
        'full_mean5': np.mean([enc[f"BiomedBERT_{s}"] for s in [42, 123, 456, 789, 1234]], 0),
        'answer_only_mean5': np.mean(
            [np.load(f"{DRIVE}/shortcut_answer_only_probs.npy")] +
            [np.load(f"{rv}/a2_answer_only_{s}_probs.npy") for s in [123, 456, 789, 1234]], 0),
        'tfidf_answer_only': np.load(f"{rv}/a9_tfidf_test_probs.npy"),
        'full_detector_seed42_saved': np.load(f"{rv}/a1_test_time_probs.npz")['full'],
    }
    r = {'items_total': int(len(diff))}
    for tol in [3, 5]:
        keep = diff <= tol
        rows = np.repeat(keep, 2)
        r[f'within_{tol}_words'] = {'items': int(keep.sum())}
        for k, p in models.items():
            r[f'within_{tol}_words'][k] = score(p[rows], y_te[rows])
    return r


# =====================================================================
# Shared model helpers
# =====================================================================
class PairDS(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.a = [(f"{q} {k}".strip() or "[EMPTY]") for q, k in zip(df['question'], df['knowledge'])]
        self.b = [(str(x).strip() or "[EMPTY]") for x in df['answer']]
        self.y = df['label'].astype(int).values
        self.tok = tok
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        e = self.tok(self.a[i], self.b[i], truncation='longest_first', max_length=512)
        e['labels'] = int(self.y[i])
        return e


def train_model(model_name, tr, seed, out_dir=None):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    args = TrainingArguments(
        output_dir=f"/content/tmp_{seed}", num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01, eval_strategy="no",
        save_strategy="no", warmup_steps=500, logging_steps=1000,
        fp16=True, seed=seed, report_to="none")
    t = Trainer(model=model, args=args, train_dataset=PairDS(tr, tok),
                data_collator=DataCollatorWithPadding(tok))
    t.train()
    if out_dir:
        model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return model.eval(), tok


@torch.no_grad()
def predict(model, tok, df, batch=64):
    model = model.cuda().eval()
    a = [(f"{q} {k}".strip() or "[EMPTY]") for q, k in zip(df['question'], df['knowledge'])]
    b = [(str(x).strip() or "[EMPTY]") for x in df['answer']]
    out = []
    for i in range(0, len(df), batch):
        e = tok(a[i:i+batch], b[i:i+batch], truncation='longest_first',
                max_length=512, padding=True, return_tensors='pt').to('cuda')
        out.append(torch.softmax(model(**e).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def shuffle_df(df, seed=0):
    d = df.copy().reset_index(drop=True); n = len(d)
    perm = np.random.default_rng(seed).permutation(n)
    while (perm == np.arange(n)).any():
        same = perm == np.arange(n); perm[same] = (perm[same] + 1) % n
    d['knowledge'] = d['knowledge'].values[perm]; d['question'] = d['question'].values[perm]
    return d


def free():
    gc.collect(); torch.cuda.empty_cache()


# =====================================================================
# P2 — answer-only BiomedBERT, trained once and saved for reuse
# =====================================================================
AO_DIR = f"{OUT}/answer_only_seed42"


def p2():
    tr = train_df.assign(question='', knowledge='')
    m, t = train_model(BIOMEDBERT, tr, 42, AO_DIR)
    r = score(predict(m, t, test_df.assign(question='', knowledge='')), y_te)
    del m; free()
    return r


# =====================================================================
# P3 — style-controlled test set
# =====================================================================
TO_ASSERTIVE = """Rewrite the answer below to a biomedical research question.

Keep the meaning and the conclusion exactly the same. Do not add or remove any fact, number, drug, gene, population, mechanism or outcome, and do not correct anything.
Change only the style: write it as one or two short, direct, declarative sentences in the third person, stating the finding plainly. Do not use first-person words (we, our, us) or phrases such as "this study", "these results", "these findings", "further studies", "may", "might", "could" or "suggest".
Aim for about {n} words.

Answer:
{answer}

Rewritten answer:"""

TO_HEDGED = """Rewrite the answer below to a biomedical research question.

Keep the meaning and every claim exactly the same, including any claim that is wrong. Do not add or remove any fact, number, drug, gene, population, mechanism or outcome, and do not correct anything.
Change only the style: write it the way the authors of a study would phrase the conclusion of their abstract, in the first person plural, with cautious academic hedging (for example "our results suggest", "may", "these findings indicate", "further studies are needed").
Aim for about {n} words.

Answer:
{answer}

Rewritten answer:"""


def rewrite_all(texts, targets, template, cache_csv, batch=24):
    done = pd.read_csv(cache_csv) if os.path.exists(cache_csv) else pd.DataFrame(columns=['i', 'text'])
    start = len(done)
    if start >= len(texts):
        return done.sort_values('i')['text'].tolist()
    tok = AutoTokenizer.from_pretrained(REWRITER); tok.padding_side = 'left'
    lm = AutoModelForCausalLM.from_pretrained(REWRITER, torch_dtype=torch.bfloat16,
                                              device_map='cuda').eval()
    rows = done.to_dict('records')
    for i in range(start, len(texts), batch):
        prompts = [tok.apply_chat_template(
            [{"role": "user", "content": template.format(answer=t, n=n)}],
            tokenize=False, add_generation_prompt=True)
            for t, n in zip(texts[i:i+batch], targets[i:i+batch])]
        enc = tok(prompts, return_tensors='pt', padding=True).to('cuda')
        with torch.no_grad():
            gen = lm.generate(**enc, max_new_tokens=160, do_sample=False,
                              pad_token_id=tok.eos_token_id)
        outs = tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        for j, o in enumerate(outs):
            o = o.strip().strip('"').strip()
            o = re.sub(r'^(Rewritten answer:\s*)', '', o, flags=re.I).strip()
            rows.append({'i': i + j, 'text': o})
        pd.DataFrame(rows).to_csv(cache_csv, index=False)
        print(f"  {os.path.basename(cache_csv)}: {min(i+batch, len(texts))}/{len(texts)}", flush=True)
    del lm; free()
    return pd.DataFrame(rows).sort_values('i')['text'].tolist()


HEDGE = r"\b(may|might|could|suggest\w*|our|we|us|these (results|findings)|this study|further (studies|research))\b"


@torch.no_grad()
def nli_entails(premises, hyps, batch=32):
    tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    m = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).cuda().eval().half()
    ent = [i for i, l in m.config.id2label.items() if l.lower() == 'entailment'][0]
    out = []
    for i in range(0, len(premises), batch):
        e = tok(premises[i:i+batch], hyps[i:i+batch], truncation=True, max_length=512,
                padding=True, return_tensors='pt').to('cuda')
        out.append(torch.softmax(m(**e).logits.float(), -1).argmax(-1).cpu().numpy() == ent)
    del m; free()
    return np.concatenate(out)


def p3():
    corr = test_df.iloc[0::2].reset_index(drop=True)
    hall = test_df.iloc[1::2].reset_index(drop=True)
    new_corr = rewrite_all(corr['answer'].tolist(), hall['answer'].map(words).tolist(),
                           TO_ASSERTIVE, f"{OUT}/p3_correct_assertive.csv")
    new_hall = rewrite_all(hall['answer'].tolist(), corr['answer'].map(words).tolist(),
                           TO_HEDGED, f"{OUT}/p3_halluc_hedged.csv")

    def interleave(c_ans, h_ans):
        d = test_df.copy()
        d.loc[d.index[0::2], 'answer'] = list(c_ans)
        d.loc[d.index[1::2], 'answer'] = list(h_ans)
        return d

    sets = {'original': test_df,
            'correct_restyled': interleave(new_corr, hall['answer']),
            'both_swapped': interleave(new_corr, new_hall)}

    r = {'style_stats': {}}
    for name, d in sets.items():
        c, h = d['answer'].iloc[0::2], d['answer'].iloc[1::2]
        r['style_stats'][name] = {
            'correct_median_words': float(c.map(words).median()),
            'halluc_median_words': float(h.map(words).median()),
            'correct_hedge_terms_per_answer': round(float(c.str.lower().str.count(HEDGE).mean()), 2),
            'halluc_hedge_terms_per_answer': round(float(h.str.lower().str.count(HEDGE).mean()), 2)}

    # meaning preservation: does the original entail the rewrite, and vice versa?
    f1 = nli_entails(corr['answer'].tolist(), list(new_corr))
    b1 = nli_entails(list(new_corr), corr['answer'].tolist())
    f2 = nli_entails(hall['answer'].tolist(), list(new_hall))
    b2 = nli_entails(list(new_hall), hall['answer'].tolist())
    r['meaning_preserved_pct'] = {
        'correct_orig_entails_rewrite': round(float(f1.mean() * 100), 1),
        'correct_mutual_entailment': round(float((f1 & b1).mean() * 100), 1),
        'halluc_orig_entails_rewrite': round(float(f2.mean() * 100), 1),
        'halluc_mutual_entailment': round(float((f2 & b2).mean() * 100), 1)}
    keep = f1 & b1 & f2 & b2
    r['items_both_rewrites_mutually_entailed'] = int(keep.sum())

    # 50 random items for Priya to read by hand
    idx = np.random.default_rng(0).choice(len(corr), 50, replace=False)
    pd.DataFrame({'item': idx, 'question': corr['question'].values[idx],
                  'correct_original': corr['answer'].values[idx],
                  'correct_rewritten': np.array(new_corr)[idx],
                  'halluc_original': hall['answer'].values[idx],
                  'halluc_rewritten': np.array(new_hall)[idx],
                  'correct_meaning_kept (Y/N)': '', 'halluc_meaning_kept (Y/N)': ''}
                 ).to_csv(f"{OUT}/p3_manual_check.csv", index=False)

    # models to score
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                          max_features=50000, lowercase=True)
    lr = LogisticRegression(max_iter=5000, C=1.0).fit(
        vec.fit_transform(train_df['answer']), train_df['label'].values)
    full_tok = AutoTokenizer.from_pretrained(f"{DRIVE}/biomedbert_v2_tok")
    full_m = AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/biomedbert_v2")
    ao_tok = AutoTokenizer.from_pretrained(AO_DIR)
    ao_m = AutoModelForSequenceClassification.from_pretrained(AO_DIR)

    rows_keep = np.repeat(keep, 2)
    for name, d in sets.items():
        res = {
            'tfidf_answer_only': score(lr.predict_proba(vec.transform(d['answer']))[:, 1], y_te),
            'biomedbert_answer_only': score(predict(ao_m, ao_tok, d.assign(question='', knowledge='')), y_te),
            'biomedbert_full': score(pf := predict(full_m, full_tok, d), y_te),
            'biomedbert_full_shuffled_evidence': score(predict(full_m, full_tok, shuffle_df(d)), y_te),
        }
        res['biomedbert_full_meaning_kept_subset'] = score(pf[rows_keep], y_te[rows_keep])
        r[name] = res
        print(name, json.dumps(res), flush=True)
    del full_m, ao_m; free()
    return r


# =====================================================================
# P4 — the same controls inside HaluEval QA (train and test on HaluEval)
# =====================================================================
def p4():
    he = load_dataset("pminervini/HaluEval", "qa")['data'].to_pandas()
    qs = he['question'].unique()
    rng = np.random.default_rng(42); rng.shuffle(qs)
    test_q = set(qs[:2000])
    rows = []
    for _, ex in he.iterrows():
        base = {'question': ex['question'], 'knowledge': ex['knowledge'],
                'split': 'test' if ex['question'] in test_q else 'train'}
        rows.append({**base, 'answer': ex['right_answer'], 'label': 0})
        rows.append({**base, 'answer': ex['hallucinated_answer'], 'label': 1})
    d = pd.DataFrame(rows)
    for c in ['question', 'knowledge', 'answer']:
        d[c] = d[c].fillna('').astype(str)
    tr = d[d.split == 'train'].reset_index(drop=True)
    te = d[d.split == 'test'].reset_index(drop=True)
    yt = te['label'].values
    r = {'n_train_pairs': int(len(tr)), 'n_test_pairs': int(len(te)),
         'question_overlap': len(set(tr.question) & set(te.question))}

    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                          max_features=50000, lowercase=True)
    lr = LogisticRegression(max_iter=5000).fit(vec.fit_transform(tr['answer']), tr['label'])
    r['tfidf_answer_only'] = score(lr.predict_proba(vec.transform(te['answer']))[:, 1], yt)
    L = lambda x: np.log1p(x['answer'].map(words).values).reshape(-1, 1)
    r['length_only'] = score(LogisticRegression().fit(L(tr), tr['label']).predict_proba(L(te))[:, 1], yt)

    conds = {'full': lambda x: x,
             'answer_only': lambda x: x.assign(question='', knowledge=''),
             'shuffled': lambda x: shuffle_df(x, 1)}
    MODEL = "roberta-base"   # general-domain corpus, general-domain encoder
    per = []
    for cond, fn in conds.items():
        for seed in [42, 123, 456]:
            cp = f"{OUT}/p4_{cond}_{seed}.json"
            if os.path.exists(cp):
                per.append(json.load(open(cp))); continue
            m, t = train_model(MODEL, fn(tr), seed)
            res = {'condition': cond, 'seed': seed, **score(predict(m, t, fn(te)), yt)}
            if cond == 'full' and seed == 42:
                res['test_time_shuffled'] = score(predict(m, t, shuffle_df(te, 2)), yt)
                res['test_time_evidence_removed'] = score(
                    predict(m, t, te.assign(question='', knowledge='')), yt)
            json.dump(res, open(cp, 'w')); per.append(res)
            print(res, flush=True)
            del m; free()
    r['per_run'] = per
    df = pd.DataFrame(per)
    r['summary'] = {c: {'macro_f1_mean': round(df[df.condition == c].macro_f1.mean(), 4),
                        'macro_f1_sd': round(df[df.condition == c].macro_f1.std(ddof=1), 4),
                        'auroc_mean': round(df[df.condition == c].auroc.mean(), 4)}
                    for c in conds}
    return r


# =====================================================================
# RUN
# =====================================================================
assert torch.cuda.is_available(), "Runtime > Change runtime type > A100 GPU"
print("GPU:", torch.cuda.get_device_name(0))
RESULTS['P0'] = cached('p0_fact_checks', p0)
RESULTS['P1'] = cached('p1_length_matched', p1)
RESULTS['P2'] = cached('p2_answer_only_saved', p2)
RESULTS['P3'] = cached('p3_style_controlled', p3)
RESULTS['P4'] = cached('p4_halueval_within', p4)

print("\n" + "=" * 64 + "\nPASTE THIS BACK\n" + "=" * 64)
print(json.dumps(RESULTS, indent=1))
print("\nALSO: open  MyDrive/medhallu_v2/review_v16/p3_manual_check.csv ,")
print("read the 50 rows, fill the two Y/N columns (did the rewrite keep the")
print("meaning?), and send the file back.")

GPU: NVIDIA A100-SXM4-40GB


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

{'condition': 'shuffled', 'seed': 123, 'macro_f1': 0.9807, 'auroc': 0.9881, 'pairwise_acc': 0.984, 'correct_flagged': 0.005}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1000,0.213320
2000,0.098485
3000,0.079240


{'condition': 'shuffled', 'seed': 456, 'macro_f1': 0.979, 'auroc': 0.9876, 'pairwise_acc': 0.9865, 'correct_flagged': 0.0095}
[p4_halueval_within] done in 1499s

PASTE THIS BACK
{
 "P0": {
  "provenance": {
   "questions_found_in_pubmedqa": 1000,
   "correct_answer_equals_long_answer": 1000,
   "n": 1000,
   "example_medhallu": "Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar str",
   "example_pubmedqa_long_answer": "Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar str"
  },
  "zero_shot_nli_candid

In [3]:
# ============================================================
# CELL 17 — How many rewrites came out in the wrong language?
#            CPU only, seconds. Run Cell 1 first.
# ============================================================
import re, pandas as pd
OUT = "/content/drive/MyDrive/medhallu_v2/review_v16"
cjk = re.compile(r"[　-鿿가-힯]")
for f in ["p3_correct_assertive.csv", "p3_halluc_hedged.csv"]:
    d = pd.read_csv(f"{OUT}/{f}")
    bad = d[d['text'].astype(str).str.contains(cjk)]
    print(f"{f}: {len(bad)} of {len(d)} contain Chinese/Japanese/Korean characters")
    print("   item numbers:", bad['i'].tolist()[:40])

p3_correct_assertive.csv: 7 of 1000 contain Chinese/Japanese/Korean characters
   item numbers: [16, 74, 219, 408, 530, 606, 797]
p3_halluc_hedged.csv: 1 of 1000 contain Chinese/Japanese/Korean characters
   item numbers: [606]


In [4]:
# ============================================================
# CELL 18 — Re-score the style-controlled test WITHOUT the 7 items
#            whose rewrites came out partly in Chinese.
# Run Cell 1 first. Any GPU (T4 is fine). ~3 minutes.
# ============================================================
import re, json, numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DRIVE = "/content/drive/MyDrive/medhallu_v2"; OUT = f"{DRIVE}/review_v16"
cjk = re.compile(r"[　-鿿가-힯]")


def load(n):
    d = pd.read_csv(f"{DRIVE}/{n}_df.csv")
    for c in ['question', 'knowledge', 'answer']:
        d[c] = d[c].fillna('').astype(str)
    return d


train_df, test_df = load("train"), load("test")
y = test_df['label'].values
nc = pd.read_csv(f"{OUT}/p3_correct_assertive.csv").sort_values('i')['text'].astype(str).tolist()
nh = pd.read_csv(f"{OUT}/p3_halluc_hedged.csv").sort_values('i')['text'].astype(str).tolist()
bad = np.array([bool(cjk.search(a)) or bool(cjk.search(b)) for a, b in zip(nc, nh)])
keep_items = ~bad
rows = np.repeat(keep_items, 2)
print("items dropped:", int(bad.sum()), "| items kept:", int(keep_items.sum()))


def inter(c, h):
    d = test_df.copy()
    d.loc[d.index[0::2], 'answer'] = list(c)
    d.loc[d.index[1::2], 'answer'] = list(h)
    return d


hall = test_df['answer'].iloc[1::2].tolist()
sets = {'original': test_df, 'correct_restyled': inter(nc, hall), 'both_swapped': inter(nc, nh)}


def score(p, yy):
    pred = (p >= .5).astype(int)
    return {'macro_f1': round(float(f1_score(yy, pred, average='macro')), 4),
            'auroc': round(float(roc_auc_score(yy, p)), 4),
            'pairwise_acc': round(float((p[1::2] > p[0::2]).mean()), 4)}


@torch.no_grad()
def predict(m, t, d, b=64):
    m = m.cuda().eval()
    a = [(f"{q} {k}".strip() or "[EMPTY]") for q, k in zip(d['question'], d['knowledge'])]
    c = [(x.strip() or "[EMPTY]") for x in d['answer']]
    out = []
    for i in range(0, len(d), b):
        e = t(a[i:i+b], c[i:i+b], truncation='longest_first', max_length=512,
              padding=True, return_tensors='pt').to('cuda')
        out.append(torch.softmax(m(**e).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def shuffle_df(df, seed=0):
    d = df.copy().reset_index(drop=True); n = len(d)
    perm = np.random.default_rng(seed).permutation(n)
    while (perm == np.arange(n)).any():
        s = perm == np.arange(n); perm[s] = (perm[s] + 1) % n
    d['knowledge'] = d['knowledge'].values[perm]; d['question'] = d['question'].values[perm]
    return d


vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50000)
lr = LogisticRegression(max_iter=5000).fit(vec.fit_transform(train_df['answer']), train_df['label'])
ft = AutoTokenizer.from_pretrained(f"{DRIVE}/biomedbert_v2_tok")
fm = AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/biomedbert_v2")
at = AutoTokenizer.from_pretrained(f"{OUT}/answer_only_seed42")
am = AutoModelForSequenceClassification.from_pretrained(f"{OUT}/answer_only_seed42")

res = {'items_dropped': int(bad.sum()), 'items_kept': int(keep_items.sum())}
for name, d in sets.items():
    P = {'tfidf_answer_only': lr.predict_proba(vec.transform(d['answer']))[:, 1],
         'biomedbert_answer_only': predict(am, at, d.assign(question='', knowledge='')),
         'biomedbert_full': predict(fm, ft, d),
         'biomedbert_full_shuffled_evidence': predict(fm, ft, shuffle_df(d))}
    res[name] = {k: score(v[rows], y[rows]) for k, v in P.items()}
json.dump(res, open(f"{OUT}/p3_without_cjk.json", "w"), indent=1)
print("\nPASTE THIS BACK\n" + json.dumps(res, indent=1))

items dropped: 7 | items kept: 993


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


PASTE THIS BACK
{
 "items_dropped": 7,
 "items_kept": 993,
 "original": {
  "tfidf_answer_only": {
   "macro_f1": 0.9067,
   "auroc": 0.9742,
   "pairwise_acc": 0.9809
  },
  "biomedbert_answer_only": {
   "macro_f1": 0.9537,
   "auroc": 0.992,
   "pairwise_acc": 0.995
  },
  "biomedbert_full": {
   "macro_f1": 0.9713,
   "auroc": 0.9963,
   "pairwise_acc": 0.993
  },
  "biomedbert_full_shuffled_evidence": {
   "macro_f1": 0.6715,
   "auroc": 0.9854,
   "pairwise_acc": 0.9849
  }
 },
 "correct_restyled": {
  "tfidf_answer_only": {
   "macro_f1": 0.6779,
   "auroc": 0.8074,
   "pairwise_acc": 0.8167
  },
  "biomedbert_answer_only": {
   "macro_f1": 0.6852,
   "auroc": 0.8781,
   "pairwise_acc": 0.8842
  },
  "biomedbert_full": {
   "macro_f1": 0.7863,
   "auroc": 0.9459,
   "pairwise_acc": 0.9527
  },
  "biomedbert_full_shuffled_evidence": {
   "macro_f1": 0.3607,
   "auroc": 0.8651,
   "pairwise_acc": 0.8671
  }
 },
 "both_swapped": {
  "tfidf_answer_only": {
   "macro_f1": 0.2085,
  

In [5]:
# ============================================================
# CELL 19 — Final experiments.  STANDALONE.  A100 runtime.
# Run Cell 1 (Drive mount) first, then this cell.  ~60-75 min.
#
#   Q1  Five-seed robustness of the test-time ablation and the
#       style-controlled test (retrains 4 full + 4 answer-only
#       BiomedBERT models; seed 42 reuses the saved models), with
#       bootstrap confidence intervals over items.
#   Q2  A prompted LLM detector (Qwen2.5-7B-Instruct, zero-shot)
#       on the original and restyled test sets, with the correct
#       passage, an unrelated passage, and no passage.
#
# Everything caches per seed / per condition under
# medhallu_v2/review_v19/. If Colab disconnects: Cell 1, then this
# cell again; finished parts are skipped.
# Prints "PASTE THIS BACK" at the end.
# ============================================================
import os, re, json, time, gc
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, roc_auc_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForCausalLM, TrainingArguments, Trainer,
                          DataCollatorWithPadding)

DRIVE = "/content/drive/MyDrive/medhallu_v2"
V16 = f"{DRIVE}/review_v16"
OUT = f"{DRIVE}/review_v19"
assert os.path.exists(f"{DRIVE}/train_df.csv"), "Run Cell 1 (Drive mount) first"
assert torch.cuda.is_available(), "Runtime > Change runtime type > A100"
os.makedirs(OUT, exist_ok=True)
BIOMEDBERT = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
LLM = "Qwen/Qwen2.5-7B-Instruct"
SEEDS = [42, 123, 456, 789, 1234]
print("GPU:", torch.cuda.get_device_name(0))


def load(n):
    d = pd.read_csv(f"{DRIVE}/{n}_df.csv")
    for c in ['question', 'knowledge', 'answer']:
        d[c] = d[c].fillna('').astype(str)
    return d


train_df, test_df = load("train"), load("test")
y_all = test_df['label'].values
assert (y_all[0::2] == 0).all() and (y_all[1::2] == 1).all()

# ---- the three test sets, restricted to the 993 clean items ----------
cjk = re.compile(r"[　-鿿가-힯]")
nc = pd.read_csv(f"{V16}/p3_correct_assertive.csv").sort_values('i')['text'].astype(str).tolist()
nh = pd.read_csv(f"{V16}/p3_halluc_hedged.csv").sort_values('i')['text'].astype(str).tolist()
keep_items = np.array([not (cjk.search(a) or cjk.search(b)) for a, b in zip(nc, nh)])
ROWS = np.repeat(keep_items, 2)
print("clean items:", int(keep_items.sum()))


def inter(c, h):
    d = test_df.copy()
    d.loc[d.index[0::2], 'answer'] = list(c)
    d.loc[d.index[1::2], 'answer'] = list(h)
    return d


SETS = {'original': test_df,
        'correct_restyled': inter(nc, test_df['answer'].iloc[1::2].tolist()),
        'both_swapped': inter(nc, nh)}


def shuffle_df(df, seed=0):
    d = df.copy().reset_index(drop=True); n = len(d)
    perm = np.random.default_rng(seed).permutation(n)
    while (perm == np.arange(n)).any():
        s = perm == np.arange(n); perm[s] = (perm[s] + 1) % n
    d['knowledge'] = d['knowledge'].values[perm]; d['question'] = d['question'].values[perm]
    return d


def score(p, y):
    p = np.asarray(p, float); pred = (p >= .5).astype(int)
    return {'macro_f1': round(float(f1_score(y, pred, average='macro')), 4),
            'auroc': round(float(roc_auc_score(y, p)), 4),
            'pairwise_acc': round(float((p[1::2] > p[0::2]).mean()), 4),
            'correct_flagged': round(float(pred[y == 0].mean()), 4)}


def free():
    gc.collect(); torch.cuda.empty_cache()


# ---- encoder helpers --------------------------------------------------
class PairDS(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.a = [(f"{q} {k}".strip() or "[EMPTY]") for q, k in zip(df['question'], df['knowledge'])]
        self.b = [(str(x).strip() or "[EMPTY]") for x in df['answer']]
        self.y = df['label'].astype(int).values; self.tok = tok
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        e = self.tok(self.a[i], self.b[i], truncation='longest_first', max_length=512)
        e['labels'] = int(self.y[i]); return e


def train(tr, seed):
    tok = AutoTokenizer.from_pretrained(BIOMEDBERT)
    m = AutoModelForSequenceClassification.from_pretrained(BIOMEDBERT, num_labels=2)
    args = TrainingArguments(output_dir=f"/content/q1_{seed}", num_train_epochs=3,
                             per_device_train_batch_size=16, learning_rate=2e-5,
                             weight_decay=0.01, eval_strategy="no", save_strategy="no",
                             warmup_steps=500, logging_steps=1000, fp16=True, seed=seed,
                             report_to="none")
    Trainer(model=m, args=args, train_dataset=PairDS(tr, tok),
            data_collator=DataCollatorWithPadding(tok)).train()
    return m.eval(), tok


@torch.no_grad()
def predict(m, t, d, b=64):
    m = m.cuda().eval()
    a = [(f"{q} {k}".strip() or "[EMPTY]") for q, k in zip(d['question'], d['knowledge'])]
    c = [(str(x).strip() or "[EMPTY]") for x in d['answer']]
    out = []
    for i in range(0, len(d), b):
        e = t(a[i:i+b], c[i:i+b], truncation='longest_first', max_length=512,
              padding=True, return_tensors='pt').to('cuda')
        out.append(torch.softmax(m(**e).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)


# =====================================================================
# Q1 — five seeds
# =====================================================================
def q1_seed(seed):
    path = f"{OUT}/q1_seed{seed}.npz"
    if os.path.exists(path):
        print(f"Q1 seed {seed}: cached"); return dict(np.load(path))
    t0 = time.time(); P = {}
    # full-input detector
    if seed == 42:
        fm = AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/biomedbert_v2")
        ft = AutoTokenizer.from_pretrained(f"{DRIVE}/biomedbert_v2_tok")
    else:
        fm, ft = train(train_df, seed)
    P['tt_full'] = predict(fm, ft, test_df)
    P['tt_removed'] = predict(fm, ft, test_df.assign(question='', knowledge=''))
    P['tt_shuffled'] = predict(fm, ft, shuffle_df(test_df))
    P['tt_answer_removed'] = predict(fm, ft, test_df.assign(answer=''))
    for name, d in SETS.items():
        P[f'{name}__full_correct'] = predict(fm, ft, d)
        P[f'{name}__full_unrelated'] = predict(fm, ft, shuffle_df(d))
    del fm; free()
    # answer-only detector
    if seed == 42:
        am = AutoModelForSequenceClassification.from_pretrained(f"{V16}/answer_only_seed42")
        at = AutoTokenizer.from_pretrained(f"{V16}/answer_only_seed42")
    else:
        am, at = train(train_df.assign(question='', knowledge=''), seed)
    for name, d in SETS.items():
        P[f'{name}__answer_only'] = predict(am, at, d.assign(question='', knowledge=''))
    del am; free()
    np.savez(path, **P)
    print(f"Q1 seed {seed}: done in {(time.time()-t0)/60:.1f} min", flush=True)
    return P


def q1():
    allp = {s: q1_seed(s) for s in SEEDS}
    res = {'test_time': {}, 'style': {}, 'gaps': {}}
    # test-time ablation, all 2000 pairs
    for k in ['tt_full', 'tt_removed', 'tt_shuffled', 'tt_answer_removed']:
        per = [score(allp[s][k], y_all) for s in SEEDS]
        res['test_time'][k] = {m: [round(float(np.mean([p[m] for p in per])), 4),
                                   round(float(np.std([p[m] for p in per], ddof=1)), 4)]
                               for m in per[0]}
    # style-controlled, 993 clean items
    y = y_all[ROWS]
    for name in SETS:
        res['style'][name] = {}
        for mdl in ['full_correct', 'full_unrelated', 'answer_only']:
            per = [score(allp[s][f'{name}__{mdl}'][ROWS], y) for s in SEEDS]
            res['style'][name][mdl] = {m: [round(float(np.mean([p[m] for p in per])), 4),
                                           round(float(np.std([p[m] for p in per], ddof=1)), 4)]
                                       for m in per[0]}
    # gap = pairwise(correct passage) - pairwise(unrelated passage),
    # per seed, and a bootstrap CI over items of the seed-averaged gap
    rng = np.random.default_rng(0)
    n_items = int(keep_items.sum())
    for name in SETS:
        wins = {}
        for mdl in ['full_correct', 'full_unrelated']:
            w = np.stack([(allp[s][f'{name}__{mdl}'][ROWS][1::2] >
                           allp[s][f'{name}__{mdl}'][ROWS][0::2]).astype(float) for s in SEEDS])
            wins[mdl] = w                      # seeds x items
        diff = wins['full_correct'] - wins['full_unrelated']
        per_seed = diff.mean(1)
        boots = [diff[:, rng.integers(0, n_items, n_items)].mean() for _ in range(5000)]
        res['gaps'][name] = {
            'gap_mean_over_seeds': round(float(per_seed.mean()), 4),
            'gap_sd_over_seeds': round(float(per_seed.std(ddof=1)), 4),
            'gap_per_seed': [round(float(g), 4) for g in per_seed],
            'bootstrap_ci95_items': [round(float(np.percentile(boots, 2.5)), 4),
                                     round(float(np.percentile(boots, 97.5)), 4)]}
    return res


# =====================================================================
# Q2 — prompted LLM detector (zero-shot, Qwen2.5-7B-Instruct)
# P(hallucinated) = P("No") / (P("Yes") + P("No")) for the next token
# =====================================================================
WITH_PASSAGE = """You are checking an answer to a biomedical research question against a source passage.

Passage:
{k}

Question:
{q}

Answer:
{a}

Is the answer correct and supported by the passage? Reply with Yes or No only."""

NO_PASSAGE = """You are checking an answer to a biomedical research question.

Question:
{q}

Answer:
{a}

Is the answer correct? Reply with Yes or No only."""


@torch.no_grad()
def llm_scores(lm, tok, d, with_passage, b=16):
    yes = tok.encode("Yes", add_special_tokens=False)[0]
    no = tok.encode("No", add_special_tokens=False)[0]
    prompts = []
    for q, k, a in zip(d['question'], d['knowledge'], d['answer']):
        k = " ".join(k.split()[:450])            # keep prompts within budget
        body = (WITH_PASSAGE.format(k=k, q=q, a=a) if with_passage
                else NO_PASSAGE.format(q=q, a=a))
        prompts.append(tok.apply_chat_template([{"role": "user", "content": body}],
                                               tokenize=False, add_generation_prompt=True))
    out = []
    for i in range(0, len(prompts), b):
        e = tok(prompts[i:i+b], return_tensors='pt', padding=True).to('cuda')
        logits = lm(**e).logits[:, -1, :].float()
        two = torch.softmax(logits[:, [yes, no]], -1)
        out.append(two[:, 1].cpu().numpy())
        if i % (b * 40) == 0:
            print(f"    {i}/{len(prompts)}", flush=True)
    return np.concatenate(out)


def q2():
    tok = AutoTokenizer.from_pretrained(LLM); tok.padding_side = 'left'
    lm = None
    y = y_all[ROWS]; res = {}
    for name, d in SETS.items():
        d = d[ROWS].reset_index(drop=True)
        res[name] = {}
        for cond, dd, wp in [('correct_passage', d, True),
                             ('unrelated_passage', shuffle_df(d), True),
                             ('no_passage', d, False)]:
            path = f"{OUT}/q2_{name}_{cond}.npy"
            if os.path.exists(path):
                p = np.load(path)
            else:
                if lm is None:
                    lm = AutoModelForCausalLM.from_pretrained(LLM, dtype=torch.bfloat16,
                                                              device_map='cuda').eval()
                print(f"  Q2 {name} / {cond}", flush=True)
                p = llm_scores(lm, tok, dd, wp); np.save(path, p)
            res[name][cond] = score(p, y)
    if lm is not None:
        del lm; free()
    return res


def cached(name, fn):
    path = f"{OUT}/{name}.json"
    if os.path.exists(path):
        print(f"[{name}] cached"); return json.load(open(path))
    r = fn(); json.dump(r, open(path, "w"), indent=1); return r


R = {'Q1': cached('q1_summary', q1), 'Q2': cached('q2_summary', q2)}
print("\n" + "=" * 64 + "\nPASTE THIS BACK\n" + "=" * 64)
print(json.dumps(R, indent=1))

GPU: NVIDIA A100-SXM4-40GB
clean items: 993


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Q1 seed 42: done in 2.1 min


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

Step,Training Loss
1000,0.293584
2000,0.137248
3000,0.084884


Q1 seed 1234: done in 8.7 min


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  Q2 original / correct_passage
    0/1986
    640/1986
    1280/1986
    1920/1986
  Q2 original / unrelated_passage
    0/1986
    640/1986
    1280/1986
    1920/1986
  Q2 original / no_passage
    0/1986
    640/1986
    1280/1986
    1920/1986
  Q2 correct_restyled / correct_passage
    0/1986
    640/1986
    1280/1986
    1920/1986
  Q2 correct_restyled / unrelated_passage
    0/1986
    640/1986
    1280/1986
    1920/1986
  Q2 correct_restyled / no_passage
    0/1986
    640/1986
    1280/1986
    1920/1986
  Q2 both_swapped / correct_passage
    0/1986
    640/1986
    1280/1986
    1920/1986
  Q2 both_swapped / unrelated_passage
    0/1986
    640/1986
    1280/1986
    1920/1986
  Q2 both_swapped / no_passage
    0/1986
    640/1986
    1280/1986
    1920/1986

PASTE THIS BACK
{
 "Q1": {
  "test_time": {
   "tt_full": {
    "macro_f1": [
     0.9676,
     0.0033
    ],
    "auroc": [
     0.9948,
     0.001
    ],
    "pairwise_acc": [
     0.9936,
     0.0009
    ],
    "c

In [6]:
# ============================================================
# CELL 20 — Independence from the Qwen family.  STANDALONE.
# A100 runtime. Run Cell 1 (Drive mount) first, then this cell.
# About 90 minutes. Resumable: re-run Cell 1 + this cell after a
# disconnect; finished parts are skipped.
#
#   R1  Rewrite the test answers with a second model family
#       (microsoft/phi-4), check language + meaning, and export
#       50 items for a manual check.
#   R2  Score the five BiomedBERT seeds (full + answer-only) and
#       TF-IDF on the Phi-4 rewrites; gaps with bootstrap CIs.
#   R3  A second prompted detector from a third family
#       (allenai/OLMo-2-1124-7B-Instruct) on all test sets, and the
#       Qwen detector on the Phi-4 rewrites.
#
# Outputs go to medhallu_v2/review_v20/. Prints "PASTE THIS BACK".
# ============================================================
import os, re, json, time, gc
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForCausalLM, TrainingArguments, Trainer,
                          DataCollatorWithPadding)

DRIVE = "/content/drive/MyDrive/medhallu_v2"
V16, V19, OUT = f"{DRIVE}/review_v16", f"{DRIVE}/review_v19", f"{DRIVE}/review_v20"
assert os.path.exists(f"{DRIVE}/train_df.csv"), "Run Cell 1 (Drive mount) first"
assert torch.cuda.is_available(), "Runtime > Change runtime type > A100"
os.makedirs(OUT, exist_ok=True)
BIOMEDBERT = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
REWRITER2 = "microsoft/phi-4"
QWEN = "Qwen/Qwen2.5-7B-Instruct"
OLMO = "allenai/OLMo-2-1124-7B-Instruct"
NLI_MODEL = "cross-encoder/nli-deberta-v3-large"
SEEDS = [42, 123, 456, 789, 1234]
print("GPU:", torch.cuda.get_device_name(0))


def load(n):
    d = pd.read_csv(f"{DRIVE}/{n}_df.csv")
    for c in ['question', 'knowledge', 'answer']:
        d[c] = d[c].fillna('').astype(str)
    return d


train_df, test_df = load("train"), load("test")
y_all = test_df['label'].values
assert (y_all[0::2] == 0).all() and (y_all[1::2] == 1).all()
NONLATIN = re.compile(r"[Ѐ-ӿ؀-ۿ　-鿿가-힯]")


def words(s): return len(str(s).split())


def free(): gc.collect(); torch.cuda.empty_cache()


def score(p, y):
    p = np.asarray(p, float); pred = (p >= .5).astype(int)
    return {'macro_f1': round(float(f1_score(y, pred, average='macro')), 4),
            'auroc': round(float(roc_auc_score(y, p)), 4),
            'pairwise_acc': round(float((p[1::2] > p[0::2]).mean()), 4),
            'correct_flagged': round(float(pred[y == 0].mean()), 4)}


def shuffle_df(df, seed=0):
    d = df.copy().reset_index(drop=True); n = len(d)
    perm = np.random.default_rng(seed).permutation(n)
    while (perm == np.arange(n)).any():
        s = perm == np.arange(n); perm[s] = (perm[s] + 1) % n
    d['knowledge'] = d['knowledge'].values[perm]; d['question'] = d['question'].values[perm]
    return d


def inter(c, h):
    d = test_df.copy()
    d.loc[d.index[0::2], 'answer'] = list(c)
    d.loc[d.index[1::2], 'answer'] = list(h)
    return d


def cached(name, fn):
    path = f"{OUT}/{name}.json"
    if os.path.exists(path):
        print(f"[{name}] cached"); return json.load(open(path))
    t0 = time.time(); r = fn()
    json.dump(r, open(path, "w"), indent=1)
    print(f"[{name}] done in {(time.time()-t0)/60:.1f} min", flush=True)
    return r


# =====================================================================
# R1 — rewrite with Phi-4 (same instructions as the Qwen rewrite)
# =====================================================================
TO_ASSERTIVE = """Rewrite the answer below to a biomedical research question.

Keep the meaning and the conclusion exactly the same. Do not add or remove any fact, number, drug, gene, population, mechanism or outcome, and do not correct anything.
Change only the style: write it as one or two short, direct, declarative sentences in the third person, stating the finding plainly. Do not use first-person words (we, our, us) or phrases such as "this study", "these results", "these findings", "further studies", "may", "might", "could" or "suggest".
Aim for about {n} words. Reply in English with the rewritten answer only.

Answer:
{answer}

Rewritten answer:"""

TO_HEDGED = """Rewrite the answer below to a biomedical research question.

Keep the meaning and every claim exactly the same, including any claim that is wrong. Do not add or remove any fact, number, drug, gene, population, mechanism or outcome, and do not correct anything.
Change only the style: write it the way the authors of a study would phrase the conclusion of their abstract, in the first person plural, with cautious academic hedging (for example "our results suggest", "may", "these findings indicate", "further studies are needed").
Aim for about {n} words. Reply in English with the rewritten answer only.

Answer:
{answer}

Rewritten answer:"""


def rewrite_all(texts, targets, template, cache_csv, batch=16):
    done = pd.read_csv(cache_csv) if os.path.exists(cache_csv) else pd.DataFrame(columns=['i', 'text'])
    if len(done) >= len(texts):
        return done.sort_values('i')['text'].astype(str).tolist()
    tok = AutoTokenizer.from_pretrained(REWRITER2); tok.padding_side = 'left'
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    lm = AutoModelForCausalLM.from_pretrained(REWRITER2, dtype=torch.bfloat16,
                                              device_map='cuda').eval()
    rows = done.to_dict('records')
    for i in range(len(done), len(texts), batch):
        prompts = [tok.apply_chat_template(
            [{"role": "user", "content": template.format(answer=t, n=n)}],
            tokenize=False, add_generation_prompt=True)
            for t, n in zip(texts[i:i+batch], targets[i:i+batch])]
        enc = tok(prompts, return_tensors='pt', padding=True).to('cuda')
        with torch.no_grad():
            gen = lm.generate(**enc, max_new_tokens=160, do_sample=False,
                              pad_token_id=tok.pad_token_id)
        outs = tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        for j, o in enumerate(outs):
            o = re.sub(r'^(Rewritten answer:\s*)', '', o.strip(), flags=re.I).strip().strip('"').strip()
            rows.append({'i': i + j, 'text': o})
        pd.DataFrame(rows).to_csv(cache_csv, index=False)
        print(f"  {os.path.basename(cache_csv)}: {min(i+batch, len(texts))}/{len(texts)}", flush=True)
    del lm; free()
    return pd.DataFrame(rows).sort_values('i')['text'].astype(str).tolist()


@torch.no_grad()
def nli_entails(prem, hyp, batch=32):
    tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    m = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).cuda().eval().half()
    ent = [i for i, l in m.config.id2label.items() if l.lower() == 'entailment'][0]
    out = []
    for i in range(0, len(prem), batch):
        e = tok(prem[i:i+batch], hyp[i:i+batch], truncation=True, max_length=512,
                padding=True, return_tensors='pt').to('cuda')
        out.append(m(**e).logits.float().argmax(-1).cpu().numpy() == ent)
    del m; free()
    return np.concatenate(out)


HEDGE = r"\b(may|might|could|suggest\w*|our|we|us|these (results|findings)|this study|further (studies|research))\b"

corr = test_df.iloc[0::2].reset_index(drop=True)
hall = test_df.iloc[1::2].reset_index(drop=True)


def r1():
    pc = rewrite_all(corr['answer'].tolist(), hall['answer'].map(words).tolist(),
                     TO_ASSERTIVE, f"{OUT}/phi_correct_assertive.csv")
    ph = rewrite_all(hall['answer'].tolist(), corr['answer'].map(words).tolist(),
                     TO_HEDGED, f"{OUT}/phi_halluc_hedged.csv")
    bad = np.array([bool(NONLATIN.search(a) or NONLATIN.search(b) or not a.strip() or not b.strip())
                    for a, b in zip(pc, ph)])
    np.save(f"{OUT}/phi_keep_items.npy", ~bad)
    ent = nli_entails(corr['answer'].tolist(), pc)
    idx = np.random.default_rng(1).choice(np.where(~bad)[0], 50, replace=False)
    pd.DataFrame({'item': idx, 'question': corr['question'].values[idx],
                  'correct_original': corr['answer'].values[idx],
                  'correct_rewritten': np.array(pc)[idx],
                  'halluc_original': hall['answer'].values[idx],
                  'halluc_rewritten': np.array(ph)[idx],
                  'correct_meaning_kept (Y/N)': '', 'halluc_meaning_kept (Y/N)': ''}
                 ).to_csv(f"{OUT}/phi_manual_check.csv", index=False)
    c = pd.Series(pc)[~bad]; h = pd.Series(ph)[~bad]
    return {'items_excluded_nonenglish_or_empty': int(bad.sum()),
            'items_kept': int((~bad).sum()),
            'correct_orig_entails_rewrite_pct': round(float(ent[~bad].mean() * 100), 1),
            'correct_rewrite_median_words': float(c.map(words).median()),
            'halluc_rewrite_median_words': float(h.map(words).median()),
            'correct_rewrite_hedges_per_answer': round(float(c.str.lower().str.count(HEDGE).mean()), 2),
            'halluc_rewrite_hedges_per_answer': round(float(h.str.lower().str.count(HEDGE).mean()), 2)}


# =====================================================================
# Shared: test sets
# =====================================================================
def phi_sets():
    pc = pd.read_csv(f"{OUT}/phi_correct_assertive.csv").sort_values('i')['text'].astype(str).tolist()
    ph = pd.read_csv(f"{OUT}/phi_halluc_hedged.csv").sort_values('i')['text'].astype(str).tolist()
    keep = np.load(f"{OUT}/phi_keep_items.npy")
    return {'phi_correct_restyled': inter(pc, hall['answer'].tolist()),
            'phi_both_swapped': inter(pc, ph)}, keep


def qwen_sets():
    cjk = re.compile(r"[　-鿿가-힯]")
    nc = pd.read_csv(f"{V16}/p3_correct_assertive.csv").sort_values('i')['text'].astype(str).tolist()
    nh = pd.read_csv(f"{V16}/p3_halluc_hedged.csv").sort_values('i')['text'].astype(str).tolist()
    keep = np.array([not (cjk.search(a) or cjk.search(b)) for a, b in zip(nc, nh)])
    return {'original': test_df,
            'correct_restyled': inter(nc, hall['answer'].tolist()),
            'both_swapped': inter(nc, nh)}, keep


# =====================================================================
# R2 — five BiomedBERT seeds + TF-IDF on the Phi-4 rewrites
# =====================================================================
class PairDS(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.a = [(f"{q} {k}".strip() or "[EMPTY]") for q, k in zip(df['question'], df['knowledge'])]
        self.b = [(str(x).strip() or "[EMPTY]") for x in df['answer']]
        self.y = df['label'].astype(int).values; self.tok = tok
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        e = self.tok(self.a[i], self.b[i], truncation='longest_first', max_length=512)
        e['labels'] = int(self.y[i]); return e


def train(tr, seed):
    tok = AutoTokenizer.from_pretrained(BIOMEDBERT)
    m = AutoModelForSequenceClassification.from_pretrained(BIOMEDBERT, num_labels=2)
    args = TrainingArguments(output_dir=f"/content/r2_{seed}", num_train_epochs=3,
                             per_device_train_batch_size=16, learning_rate=2e-5,
                             weight_decay=0.01, eval_strategy="no", save_strategy="no",
                             warmup_steps=500, logging_steps=1000, fp16=True, seed=seed,
                             report_to="none")
    Trainer(model=m, args=args, train_dataset=PairDS(tr, tok),
            data_collator=DataCollatorWithPadding(tok)).train()
    return m.eval(), tok


@torch.no_grad()
def predict(m, t, d, b=64):
    m = m.cuda().eval()
    a = [(f"{q} {k}".strip() or "[EMPTY]") for q, k in zip(d['question'], d['knowledge'])]
    c = [(str(x).strip() or "[EMPTY]") for x in d['answer']]
    out = []
    for i in range(0, len(d), b):
        e = t(a[i:i+b], c[i:i+b], truncation='longest_first', max_length=512,
              padding=True, return_tensors='pt').to('cuda')
        out.append(torch.softmax(m(**e).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def r2_seed(seed, SETS):
    path = f"{OUT}/r2_seed{seed}.npz"
    if os.path.exists(path):
        print(f"R2 seed {seed}: cached"); return dict(np.load(path))
    t0 = time.time(); P = {}
    if seed == 42:
        fm = AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/biomedbert_v2")
        ft = AutoTokenizer.from_pretrained(f"{DRIVE}/biomedbert_v2_tok")
    else:
        fm, ft = train(train_df, seed)
    for name, d in SETS.items():
        P[f'{name}__full_correct'] = predict(fm, ft, d)
        P[f'{name}__full_unrelated'] = predict(fm, ft, shuffle_df(d))
    del fm; free()
    if seed == 42:
        am = AutoModelForSequenceClassification.from_pretrained(f"{V16}/answer_only_seed42")
        at = AutoTokenizer.from_pretrained(f"{V16}/answer_only_seed42")
    else:
        am, at = train(train_df.assign(question='', knowledge=''), seed)
    for name, d in SETS.items():
        P[f'{name}__answer_only'] = predict(am, at, d.assign(question='', knowledge=''))
    del am; free()
    np.savez(path, **P)
    print(f"R2 seed {seed}: done in {(time.time()-t0)/60:.1f} min", flush=True)
    return P


def r2():
    SETS, keep = phi_sets()
    rows = np.repeat(keep, 2); y = y_all[rows]; n_items = int(keep.sum())
    allp = {s: r2_seed(s, SETS) for s in SEEDS}
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50000)
    lr = LogisticRegression(max_iter=5000).fit(vec.fit_transform(train_df['answer']), train_df['label'])
    res = {'items': n_items}
    rng = np.random.default_rng(0)
    for name, d in SETS.items():
        r = {'tfidf_answer_only': score(lr.predict_proba(vec.transform(d['answer']))[:, 1][rows], y)}
        for mdl in ['full_correct', 'full_unrelated', 'answer_only']:
            per = [score(allp[s][f'{name}__{mdl}'][rows], y) for s in SEEDS]
            r[mdl] = {m: [round(float(np.mean([p[m] for p in per])), 4),
                          round(float(np.std([p[m] for p in per], ddof=1)), 4)] for m in per[0]}
        w = {m: np.stack([(allp[s][f'{name}__{m}'][rows][1::2] > allp[s][f'{name}__{m}'][rows][0::2]
                           ).astype(float) for s in SEEDS]) for m in ['full_correct', 'full_unrelated']}
        diff = w['full_correct'] - w['full_unrelated']
        boots = [diff[:, rng.integers(0, n_items, n_items)].mean() for _ in range(5000)]
        r['gap'] = {'mean': round(float(diff.mean(1).mean()), 4),
                    'sd_over_seeds': round(float(diff.mean(1).std(ddof=1)), 4),
                    'per_seed': [round(float(g), 4) for g in diff.mean(1)],
                    'ci95': [round(float(np.percentile(boots, 2.5)), 4),
                             round(float(np.percentile(boots, 97.5)), 4)]}
        res[name] = r
    return res


# =====================================================================
# R3 — prompted detectors: OLMo-2 on everything, Qwen on Phi-4 sets
# =====================================================================
WITH_PASSAGE = """You are checking an answer to a biomedical research question against a source passage.

Passage:
{k}

Question:
{q}

Answer:
{a}

Is the answer correct and supported by the passage? Reply with Yes or No only."""

NO_PASSAGE = """You are checking an answer to a biomedical research question.

Question:
{q}

Answer:
{a}

Is the answer correct? Reply with Yes or No only."""


@torch.no_grad()
def llm_scores(lm, tok, d, with_passage, b=16):
    yes = tok.encode("Yes", add_special_tokens=False)[0]
    no = tok.encode("No", add_special_tokens=False)[0]
    prompts = []
    for q, k, a in zip(d['question'], d['knowledge'], d['answer']):
        k = " ".join(k.split()[:450])
        body = WITH_PASSAGE.format(k=k, q=q, a=a) if with_passage else NO_PASSAGE.format(q=q, a=a)
        prompts.append(tok.apply_chat_template([{"role": "user", "content": body}],
                                               tokenize=False, add_generation_prompt=True))
    out = []
    for i in range(0, len(prompts), b):
        e = tok(prompts[i:i+b], return_tensors='pt', padding=True,
                add_special_tokens=False).to('cuda')
        logits = lm(**e).logits[:, -1, :].float()
        out.append(torch.softmax(logits[:, [yes, no]], -1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def run_llm(model_name, tag, sets_and_keep):
    tok = AutoTokenizer.from_pretrained(model_name); tok.padding_side = 'left'
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    lm = None; res = {}
    for SETS, keep in sets_and_keep:
        rows = np.repeat(keep, 2); y = y_all[rows]
        for name, d in SETS.items():
            dd = d[rows].reset_index(drop=True); res[name] = {}
            for cond, x, wp in [('correct_passage', dd, True),
                                ('unrelated_passage', shuffle_df(dd), True),
                                ('no_passage', dd, False)]:
                path = f"{OUT}/r3_{tag}_{name}_{cond}.npy"
                if os.path.exists(path):
                    p = np.load(path)
                else:
                    if lm is None:
                        lm = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16,
                                                                  device_map='cuda').eval()
                    print(f"  R3 {tag} {name} / {cond}", flush=True)
                    p = llm_scores(lm, tok, x, wp); np.save(path, p)
                res[name][cond] = score(p, y)
    if lm is not None:
        del lm; free()
    return res


def r3():
    return {'olmo2_7b': run_llm(OLMO, 'olmo', [qwen_sets(), phi_sets()]),
            'qwen2.5_7b_on_phi_sets': run_llm(QWEN, 'qwen', [phi_sets()])}


R = {'R1': cached('r1_phi_rewrite', r1), 'R2': cached('r2_bert_on_phi', r2),
     'R3': cached('r3_prompted', r3)}
print("\n" + "=" * 64 + "\nPASTE THIS BACK\n" + "=" * 64)
print(json.dumps(R, indent=1))
print("\nALSO: MyDrive/medhallu_v2/review_v20/phi_manual_check.csv -> fill Y/N, send back.")

GPU: NVIDIA A100-SXM4-40GB


config.json:   0%|          | 0.00/802 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.25M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

  R3 olmo original / correct_passage
  R3 olmo original / unrelated_passage
  R3 olmo original / no_passage
  R3 olmo correct_restyled / correct_passage
  R3 olmo correct_restyled / unrelated_passage
  R3 olmo correct_restyled / no_passage
  R3 olmo both_swapped / correct_passage
  R3 olmo both_swapped / unrelated_passage
  R3 olmo both_swapped / no_passage
  R3 olmo phi_correct_restyled / correct_passage
  R3 olmo phi_correct_restyled / unrelated_passage
  R3 olmo phi_correct_restyled / no_passage
  R3 olmo phi_both_swapped / correct_passage
  R3 olmo phi_both_swapped / unrelated_passage
  R3 olmo phi_both_swapped / no_passage


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  R3 qwen phi_correct_restyled / correct_passage
  R3 qwen phi_correct_restyled / unrelated_passage
  R3 qwen phi_correct_restyled / no_passage
  R3 qwen phi_both_swapped / correct_passage
  R3 qwen phi_both_swapped / unrelated_passage
  R3 qwen phi_both_swapped / no_passage
[r3_prompted] done in 32.5 min

PASTE THIS BACK
{
 "R1": {
  "items_excluded_nonenglish_or_empty": 0,
  "items_kept": 1000,
  "correct_orig_entails_rewrite_pct": 98.3,
  "correct_rewrite_median_words": 26.0,
  "halluc_rewrite_median_words": 40.0,
  "correct_rewrite_hedges_per_answer": 0.13,
  "halluc_rewrite_hedges_per_answer": 4.5
 },
 "R2": {
  "items": 1000,
  "phi_correct_restyled": {
   "tfidf_answer_only": {
    "macro_f1": 0.7245,
    "auroc": 0.8411,
    "pairwise_acc": 0.853,
    "correct_flagged": 0.405
   },
   "full_correct": {
    "macro_f1": [
     0.8446,
     0.0115
    ],
    "auroc": [
     0.9639,
     0.0029
    ],
    "pairwise_acc": [
     0.9734,
     0.0029
    ],
    "correct_flagged": [
  